# Monthly Synthesis Diagnostics: Snow DA, Soil Moisture, Hydrology, and Energy Pathways

Exploratory monthly-to-seasonal diagnostics for linking the snow and soil-moisture components of the GEOS LDAS M21C multi-sensor reanalysis.

The notebook uses monthly M36 tile-space products, including raw-derived cumulative prognostic increments and raw-derived diagnostic ANA-FCST activity metrics. It tests whether snow DA propagates into hydrology, whether prior snow DA activity is associated with later soil-moisture DA activity, and whether the combined DA system affects hydrologic and energy partitioning.

Important interpretation guardrails:

- Positive signed model response is `DA - OL`.
- Raw `catch_progn_incr` products are cumulative water/mass increments and may be seasonally summed.
- Raw `inst3_1d_lndfcstana_Nt` diagnostic products are state corrections and should be averaged/RMSed, not summed as water.
- `CATDEF` is a deficit variable; `soil_water_net_approx = SRFEXC + RZEXC - CATDEF` is approximate.
- These are model-internal DA-impact diagnostics, not independent validation metrics.

Primary outputs are written to `projects/M21C_ls/output/monthly_synthesis_diagnostics/`.


## Diagnostic Plan

1. Inventory monthly files, variables, units, time coverage, and tile metadata.
2. Build reusable seasonal/monthly aggregation helpers with explicit unit handling.
3. Define NH seasonal-snow, high snow-DA-activity, and warm snow-free masks.
4. Run Analysis A: MODIS-only snow DA carryover into soil moisture and hydrology.
5. Run Analysis B: snow DA activity versus later soil-moisture DA activity.
6. Run Analysis C: joint hydrology and energy partitioning diagnostics.
7. Run Analysis D: observing-system evolution of DA activity and propagated responses.
8. Run Analysis E: water-budget plausibility checks.
9. Summarize which diagnostics are manuscript candidates, supplemental only, or not robust.


In [ ]:
from pathlib import Path
import os
import sys
import warnings

os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/private/tmp")

import numpy as np
import pandas as pd
import xarray as xr

xr.set_options(keep_attrs=True)
pd.options.display.max_columns = 160
pd.options.display.max_rows = 120

try:
    from IPython import get_ipython
    from IPython.display import display
except Exception:
    def get_ipython():
        return None
    def display(obj):
        print(obj)


def running_in_notebook() -> bool:
    shell = get_ipython()
    return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"


IN_NOTEBOOK = running_in_notebook()

import matplotlib
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception as exc:
    HAS_CARTOPY = False
    print("Cartopy unavailable; map cells will fall back to lon/lat scatter:", exc)


def show_figure(fig):
    if IN_NOTEBOOK:
        display(fig)
    plt.close(fig)


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / ".git").exists() and (p / "common/python/io/read_GEOSldas.py").exists():
            return p
    raise FileNotFoundError("Could not locate geosldas-analysis repo root")


HERE = Path.cwd().resolve()
REPO_ROOT = find_repo_root(HERE)
PROJECT_ROOT = REPO_ROOT / "projects/M21C_ls"
OUT_DIR = PROJECT_ROOT / "output/monthly_synthesis_diagnostics"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2")

INPUTS = {
    "tilecoord": DATA_DIR / "LS_OLv8_M36.ldas_tilecoord.bin",
    "catch_raw_cumulative": DATA_DIR / "catch_progn_raw_monthly_cumulative_200006_202405.nc",
    "inst3_raw_diagnostic": DATA_DIR / "inst3_fcstana_raw_monthly_diagnostic_200006_202405.nc",
    "monthly_increment_legacy": DATA_DIR / "LS_monthly_increments_2000_2024.nc",
}

MONTHLY_GROUPS = {
    "land": {
        "ol": DATA_DIR / "OLv8_land_variables_2000_2024_compressed.nc",
        "da": DATA_DIR / "DAv8_land_variables_2000_2024_compressed.nc",
    },
    "flux_core": {
        "ol": DATA_DIR / "OLv8_flux_core_2000_2024_compressed.nc",
        "da": DATA_DIR / "DAv8_flux_core_2000_2024_compressed.nc",
    },
    "latent_components": {
        "ol": DATA_DIR / "OLv8_latent_components_2000_2024_compressed.nc",
        "da": DATA_DIR / "DAv8_latent_components_2000_2024_compressed.nc",
    },
    "water_budget": {
        "ol": DATA_DIR / "OLv8_water_budget_2000_2024_compressed.nc",
        "da": DATA_DIR / "DAv8_water_budget_2000_2024_compressed.nc",
    },
    "energy_context": {
        "ol": DATA_DIR / "OLv8_energy_context_2000_2024_compressed.nc",
        "da": DATA_DIR / "DAv8_energy_context_2000_2024_compressed.nc",
    },
}

PERIODS = {
    "modis_only": {
        "label": "MODIS-only / snow-only period",
        "start": "2000-06-01",
        "end": "2007-05-31",
        "clean_years": list(range(2001, 2007)),
        "color": "#7b3294",
    },
    "pre_smap_mw": {
        "label": "microwave pre-SMAP period",
        "start": "2007-06-01",
        "end": "2015-03-31",
        "clean_years": list(range(2008, 2015)),
        "color": "#008837",
    },
    "smap_era": {
        "label": "SMAP-era microwave period",
        "start": "2015-04-01",
        "end": "2024-05-31",
        "clean_years": list(range(2016, 2024)),
        "color": "#2b6cb0",
    },
}

SEASON_WINDOWS = {
    "DJF": [12, 1, 2],
    "FMA": [2, 3, 4],
    "MAM": [3, 4, 5],
    "AMJ": [4, 5, 6],
    "MJJ": [5, 6, 7],
    "JJA": [6, 7, 8],
}

STATE_MEAN_VARS = ["SFMC", "RZMC", "FRLANDSNO", "TSOIL1", "SNOMASLAND", "SNODPLAND", "TWLAND"]
WATER_FLUX_VARS = ["PRECTOTCORRLAND", "EVLAND", "RUNSURFLAND", "BASEFLOWLAND", "QINFILLAND", "SMLAND", "WCHANGELAND"]
ENERGY_MEAN_VARS = ["LHLAND", "SHLAND", "LHLANDTRNS", "LHLANDSOIL", "LHLANDINTR", "LHLANDSBLN", "SWLAND", "LWLAND", "GHLAND"]
CATCH_SUM_VARS = ["snow_net", "snow_abs_netpack", "snow_abs_layers", "soil_water_abs_activity", "soil_water_net_approx", "srfexc_net", "rzexc_net", "catdef_net"]
CATCH_COUNT_VARS = ["snow_event_count", "time_step_count"]
INST3_DIAG_PREFIXES = ["SFMC", "RZMC", "PRMC", "TSURF", "TSOIL1"]
INST3_DIAG_SUFFIXES = ["INC_MEAN", "INC_ABS_MEAN", "INC_RMS"]
INST3_DIAG_VARS = [f"{p}_{s}" for p in INST3_DIAG_PREFIXES for s in INST3_DIAG_SUFFIXES]

DERIVED_HYDRO_VARS = ["TOTAL_RUNOFF", "RUNOFF_RATIO", "BASEFLOW_FRACTION", "MELT_INFIL_FRACTION", "ET_OVER_P"]
DERIVED_ENERGY_VARS = ["EF", "BOWEN", "TRANS_FRACTION", "SOIL_EVAP_FRACTION", "INTR_FRACTION", "SBLN_FRACTION"]
DERIVED_VARS = DERIVED_HYDRO_VARS + DERIVED_ENERGY_VARS

SNOW_POSSIBLE_SCF_THRESHOLD = 0.05
SNOW_POSSIBLE_SWE_THRESHOLD = 5.0
PERMANENT_SNOW_JJA_MEAN_MAX = 0.20
NH_MIN_LAT = 20.0
WARM_SNOWFREE_SCF_MAX = 0.05
WARM_SNOWFREE_TSOIL_MIN_K = 277.15
N_BINS = 8

print("REPO_ROOT:", REPO_ROOT)
print("OUT_DIR:", OUT_DIR)
print("DATA_DIR exists:", DATA_DIR.exists())
print("IN_NOTEBOOK:", IN_NOTEBOOK)


In [ ]:
def describe_dataset(path: Path) -> pd.DataFrame:
    rows = []
    if not path.exists():
        return pd.DataFrame([{"path": str(path), "exists": False, "file": path.name}])
    if path.suffix not in {".nc", ".nc4"}:
        return pd.DataFrame([{
            "path": str(path),
            "exists": True,
            "file": path.name,
            "variable": "file",
            "dims": "",
            "shape": "",
            "dtype": path.suffix.lstrip("."),
            "units": "",
            "long_name": "non-NetCDF auxiliary input",
        }])
    with xr.open_dataset(path) as ds:
        for name, da in ds.data_vars.items():
            rows.append({
                "path": str(path),
                "exists": True,
                "file": path.name,
                "variable": name,
                "dims": ",".join(da.dims),
                "shape": "x".join(str(da.sizes[d]) for d in da.dims),
                "dtype": str(da.dtype),
                "units": da.attrs.get("units", ""),
                "long_name": da.attrs.get("long_name", da.attrs.get("standard_name", "")),
            })
        for name in ds.coords:
            if name not in ds.data_vars:
                da = ds[name]
                rows.append({
                    "path": str(path),
                    "exists": True,
                    "file": path.name,
                    "variable": f"coord:{name}",
                    "dims": ",".join(da.dims),
                    "shape": "x".join(str(da.sizes[d]) for d in da.dims),
                    "dtype": str(da.dtype),
                    "units": da.attrs.get("units", ""),
                    "long_name": da.attrs.get("long_name", ""),
                })
    return pd.DataFrame(rows)

all_input_paths = list(INPUTS.values()) + [p for group in MONTHLY_GROUPS.values() for p in group.values()]
inventory = pd.concat([describe_dataset(path) for path in all_input_paths], ignore_index=True)
inventory.to_csv(OUT_DIR / "monthly_synthesis_input_inventory.csv", index=False)
display(inventory)

missing_files = inventory.loc[inventory["exists"].eq(False), "path"].dropna().tolist()
if missing_files:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing_files))

DATASETS = {"ol": {}, "da": {}}
for group_name, paths in MONTHLY_GROUPS.items():
    DATASETS["ol"][group_name] = xr.open_dataset(paths["ol"], decode_times=True)
    DATASETS["da"][group_name] = xr.open_dataset(paths["da"], decode_times=True)

CATCH = xr.open_dataset(INPUTS["catch_raw_cumulative"], decode_times=True)
INST3 = xr.open_dataset(INPUTS["inst3_raw_diagnostic"], decode_times=True)
LEGACY_INC = xr.open_dataset(INPUTS["monthly_increment_legacy"], decode_times=True) if INPUTS["monthly_increment_legacy"].exists() else None

ol_land = DATASETS["ol"]["land"]
da_land = DATASETS["da"]["land"]
if not np.array_equal(ol_land.time.values, da_land.time.values):
    raise ValueError("OL and DA land monthly time coordinates differ")

n_tile = ol_land.sizes["tile"]
tile_coord = ol_land["tile"] if "tile" in ol_land.coords else xr.DataArray(np.arange(n_tile), dims="tile", name="tile")
lat = (ol_land["lat"] if "lat" in ol_land else CATCH["lat"]).load()
lon = (ol_land["lon"] if "lon" in ol_land else CATCH["lon"]).load()

VAR_SOURCE = {}
for group_name, ds in DATASETS["ol"].items():
    da_ds = DATASETS["da"][group_name]
    if not np.array_equal(ds.time.values, ol_land.time.values):
        raise ValueError(f"{group_name} OL time coordinate differs from land group")
    if not np.array_equal(da_ds.time.values, da_land.time.values):
        raise ValueError(f"{group_name} DA time coordinate differs from land group")
    if ds.sizes.get("tile") != n_tile or da_ds.sizes.get("tile") != n_tile:
        raise ValueError(f"{group_name} tile dimension differs from land group")
    for var_name in ds.data_vars:
        if var_name in da_ds.data_vars and ds[var_name].dims == da_ds[var_name].dims:
            VAR_SOURCE[var_name] = group_name

sys.path.insert(0, str(REPO_ROOT / "common/python/io"))
use_area_weights = False
try:
    from read_GEOSldas import read_tilecoord
    tc = read_tilecoord(str(INPUTS["tilecoord"]))
    area_values = np.asarray(tc["area"], dtype="float64")
    if area_values.size != n_tile:
        raise ValueError(f"tilecoord area length {area_values.size} != tile dimension {n_tile}")
    tile_weight = xr.DataArray(area_values, dims="tile", coords={"tile": tile_coord}, name="tile_area")
    tile_weight = tile_weight.where(np.isfinite(tile_weight) & (tile_weight > 0))
    use_area_weights = bool(np.isfinite(tile_weight).any())
except Exception as exc:
    warnings.warn(f"Could not load tilecoord area weights; falling back to unweighted tile means: {exc}")
    tile_weight = xr.ones_like(lat, dtype="float64").rename("tile_weight_unweighted")

time_coverage_rows = []
for name, ds in [
    ("ol_land", ol_land),
    ("da_land", da_land),
    ("catch_raw_cumulative", CATCH),
    ("inst3_raw_diagnostic", INST3),
] + [(f"ol_{k}", v) for k, v in DATASETS["ol"].items()] + [(f"da_{k}", v) for k, v in DATASETS["da"].items()]:
    t = pd.to_datetime(ds.time.values)
    time_coverage_rows.append({"dataset": name, "n_time": len(t), "start": str(t.min().date()), "end": str(t.max().date()), "n_tile": ds.sizes.get("tile", np.nan)})
time_coverage = pd.DataFrame(time_coverage_rows).drop_duplicates()
time_coverage.to_csv(OUT_DIR / "monthly_synthesis_time_coverage.csv", index=False)
display(time_coverage)

registry_rows = []
for var in STATE_MEAN_VARS + WATER_FLUX_VARS + ENERGY_MEAN_VARS + DERIVED_VARS:
    registry_rows.append({"variable": var, "kind": "model_or_derived", "source": VAR_SOURCE.get(var, "derived" if var in DERIVED_VARS else "missing")})
for var in CATCH_SUM_VARS + CATCH_COUNT_VARS:
    registry_rows.append({"variable": var, "kind": "cumulative_prognostic_increment", "source": "catch_raw_cumulative" if var in CATCH else "missing"})
for var in INST3_DIAG_VARS:
    registry_rows.append({"variable": var, "kind": "diagnostic_ana_fcst_activity", "source": "inst3_raw_diagnostic" if var in INST3 else "missing"})
variable_registry = pd.DataFrame(registry_rows)
variable_registry.to_csv(OUT_DIR / "monthly_synthesis_variable_registry.csv", index=False)
display(variable_registry)

print("Area weighting:", "tilecoord area" if use_area_weights else "unweighted fallback")


## Helper Functions

These helpers enforce the unit rules used throughout the notebook.

- State and energy variables are day-weighted seasonal means.
- Water flux variables in `kg m-2 s-1` are converted to monthly totals before seasonal sums.
- Cumulative prognostic increments are summed over months.
- Diagnostic ANA-FCST activity variables are day-weighted seasonal means.
- Derived ratios are computed separately for DA and OL, then differenced.


### Figure and Aggregation Conventions

The notes beside each plotting cell describe the corresponding figure(s). A few conventions apply throughout. Model-response variables are `DA - OL`, so positive values mean the data-assimilation run is larger than the open-loop run. Cumulative prognostic increment variables such as `snow_net`, `snow_abs_netpack`, and `soil_water_abs_activity` are summed across months. Diagnostic ANA-FCST activity variables such as `RZMC_INC_RMS` are seasonal means, not water sums. Seasonal state and energy variables are day-weighted means; water fluxes in `kg m-2 s-1` are converted to monthly totals using seconds per month and then summed for seasonal totals. Domain time series and composites use tile-area weights when available. Binned-line figures use tile-year samples, quantile bins, bin-mean x/y values, and standard-error bars.


In [ ]:
def savefig(fig, stem: str):
    png = FIG_DIR / f"{stem}.png"
    pdf = FIG_DIR / f"{stem}.pdf"
    fig.savefig(png, dpi=180, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print("saved", png)
    print("saved", pdf)


def add_geo_base(ax, extent=None):
    if HAS_CARTOPY:
        ax.add_feature(cfeature.OCEAN, facecolor="white", edgecolor="none", zorder=0)
        ax.add_feature(cfeature.LAND, facecolor="0.93", edgecolor="none", zorder=0)
        ax.coastlines(linewidth=0.45, color="0.35")
        if extent is not None:
            ax.set_extent(extent, crs=ccrs.PlateCarree())
    else:
        ax.grid(True, alpha=0.25)
        if extent is not None:
            ax.set_xlim(extent[0], extent[1])
            ax.set_ylim(extent[2], extent[3])


def tile_scatter_map(ax, values, mask=None, title="", cmap="RdBu_r", norm=None, s=0.7, extent=None):
    values = np.asarray(values)
    valid = np.isfinite(lat.values) & np.isfinite(lon.values) & np.isfinite(values)
    if mask is not None:
        valid &= np.asarray(mask, dtype=bool)
    if HAS_CARTOPY:
        add_geo_base(ax, extent=extent)
        im = ax.scatter(lon.values[valid], lat.values[valid], c=values[valid], s=s, marker="s", cmap=cmap, norm=norm,
                        linewidths=0, transform=ccrs.PlateCarree(), rasterized=True)
    else:
        add_geo_base(ax, extent=extent)
        im = ax.scatter(lon.values[valid], lat.values[valid], c=values[valid], s=s, marker="s", cmap=cmap, norm=norm,
                        linewidths=0, rasterized=True)
    ax.set_title(title, fontsize=9)
    return im


def symmetric_norm(values, percentile=98.0, floor=1.0e-10):
    arr = np.asarray(values)
    finite = arr[np.isfinite(arr)]
    vmax = np.nanpercentile(np.abs(finite), percentile) if finite.size else floor
    vmax = max(float(vmax), floor)
    return mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)


def positive_norm(values, percentile=98.0, floor=1.0e-10):
    arr = np.asarray(values)
    finite = arr[np.isfinite(arr)]
    vmax = np.nanpercentile(finite, percentile) if finite.size else floor
    vmax = max(float(vmax), floor)
    return mcolors.Normalize(vmin=0.0, vmax=vmax)


FLUX_TOTAL_NAMES = {
    "prectotcorrland", "evland", "runsurfland", "baseflowland", "qinfilland",
    "smland", "wchangeland", "total_runoff", "runoff", "e", "p",
}

EXACT_UNITS = {
    "p": "kg m-2",
    "e": "kg m-2",
    "residual_assuming_dp_zero": "kg m-2",
    "residual_p_minus_e_runoff_wchange": "kg m-2",
}

UNIT_TOKENS = [
    ("rzmc_inc", "m3 m-3"),
    ("sfmc_inc", "m3 m-3"),
    ("prmc_inc", "m3 m-3"),
    ("tsurf_inc", "K"),
    ("tsoil1_inc", "K"),
    ("soil_water_abs_activity", "kg m-2"),
    ("soil_water_net_approx", "kg m-2"),
    ("srfexc_net", "kg m-2"),
    ("rzexc_net", "kg m-2"),
    ("catdef_net", "kg m-2"),
    ("snow_event_count", "count"),
    ("time_step_count", "count"),
    ("snow_abs_netpack", "kg m-2"),
    ("snow_abs_layers", "kg m-2"),
    ("snow_net", "kg m-2"),
    ("total_runoff", "kg m-2"),
    ("runsurfland", "kg m-2"),
    ("baseflowland", "kg m-2"),
    ("qinfilland", "kg m-2"),
    ("smland", "kg m-2"),
    ("evland", "kg m-2"),
    ("prectotcorrland", "kg m-2"),
    ("wchangeland", "kg m-2"),
    ("twland", "kg m-2"),
    ("snomasland", "kg m-2"),
    ("sfmc", "m3 m-3"),
    ("rzmc", "m3 m-3"),
    ("prmc", "m3 m-3"),
    ("frlandsno", "fraction"),
    ("snodpland", "m"),
    ("lhlandtrns", "W m-2"),
    ("lhlandsoil", "W m-2"),
    ("lhlandintr", "W m-2"),
    ("lhlandsbln", "W m-2"),
    ("lhland", "W m-2"),
    ("shland", "W m-2"),
    ("swland", "W m-2"),
    ("lwland", "W m-2"),
    ("ghland", "W m-2"),
    ("tsoil1", "K"),
    ("tsurf", "K"),
    ("baseflow_fraction", "fraction"),
    ("melt_infil_fraction", "fraction"),
    ("trans_fraction", "fraction"),
    ("soil_evap_fraction", "fraction"),
    ("intr_fraction", "fraction"),
    ("sbln_fraction", "fraction"),
    ("runoff_ratio", "ratio"),
    ("et_over_p", "ratio"),
    ("bowen", "ratio"),
    ("ef", "fraction"),
]

DISPLAY_LABELS = {
    "p": "precipitation",
    "e": "evapotranspiration",
    "sfmc": "SFMC",
    "rzmc": "RZMC",
    "prmc": "PRMC",
    "frlandsno": "snow-covered fraction",
    "snomasland": "snow mass",
    "snodpland": "snow depth",
    "tsoil1": "soil temperature layer 1",
    "twland": "total water storage",
    "smland": "snowmelt",
    "qinfilland": "infiltration",
    "evland": "ET",
    "runsurfland": "surface runoff",
    "baseflowland": "baseflow",
    "total_runoff": "total runoff",
    "wchangeland": "water-storage tendency",
    "lhland": "latent heat",
    "shland": "sensible heat",
    "ghland": "ground heat",
    "swland": "net shortwave",
    "lwland": "net longwave",
    "lhlandtrns": "transpiration LH",
    "lhlandsoil": "bare-soil evaporation LH",
    "lhlandintr": "interception evaporation LH",
    "lhlandsbln": "snow sublimation LH",
    "ef": "evaporative fraction",
    "snow_net": "snow net increment",
    "snow_abs_netpack": "snow DA activity",
    "soil_water_abs_activity": "soil-water DA activity",
    "soil_water_net_approx": "approx. soil-water net increment",
    "rzmc_inc_rms": "RZMC ANA-FCST RMS",
    "rzmc_inc_mean": "RZMC ANA-FCST mean",
    "rzmc_inc_abs_mean": "RZMC ANA-FCST mean abs",
    "sfmc_inc_rms": "SFMC ANA-FCST RMS",
    "sfmc_inc_mean": "SFMC ANA-FCST mean",
    "sfmc_inc_abs_mean": "SFMC ANA-FCST mean abs",
    "prmc_inc_rms": "PRMC ANA-FCST RMS",
    "prmc_inc_mean": "PRMC ANA-FCST mean",
    "residual_assuming_dp_zero": "budget residual, dP=0",
}


def normalized_metric_name(metric):
    name = str(metric).strip().lower().replace("-", "_")
    for suffix in ["_anom"]:
        if name.endswith(suffix):
            name = name[: -len(suffix)]
    for season in [s.lower() for s in SEASON_WINDOWS]:
        if name.endswith(f"_{season}"):
            name = name[: -(len(season) + 1)]
            break
    while name.startswith("abs_"):
        name = name[4:]
    if name.startswith("d") and len(name) > 1 and not name.startswith("da"):
        name = name[1:]
    return name


def metric_units(metric, context=None):
    raw = str(metric).strip().lower()
    no_abs = raw
    while no_abs.startswith("abs_"):
        no_abs = no_abs[4:]
    if no_abs.startswith("pct_"):
        return "%"

    norm = normalized_metric_name(metric)
    if norm in EXACT_UNITS:
        unit = EXACT_UNITS[norm]
    else:
        unit = ""
        for token, token_unit in UNIT_TOKENS:
            if token in norm:
                unit = token_unit
                break
    if context in {"monthly", "seasonal"} and norm in FLUX_TOTAL_NAMES:
        return "kg m-2 month-1" if context == "monthly" else "kg m-2 season-1"
    return unit


def metric_label(metric, include_units=False, context=None):
    raw = str(metric)
    raw_lower = raw.lower()
    clean_raw = raw_lower
    is_abs = False
    while clean_raw.startswith("abs_"):
        is_abs = True
        clean_raw = clean_raw[4:]
    if clean_raw.startswith("pct_"):
        clean_metric = clean_raw[4:]
        norm = normalized_metric_name(clean_metric)
        label = f"{DISPLAY_LABELS.get(norm, clean_metric.replace('_', ' '))} % change vs OL"
        if is_abs:
            label = f"abs {label}"
        if raw_lower.endswith("_anom"):
            label = f"{label} anomaly"
        if include_units:
            unit = metric_units(raw, context=context)
            if unit:
                label = f"{label} ({unit})"
        return label

    norm = normalized_metric_name(raw)
    label = DISPLAY_LABELS.get(norm, raw.replace("_", " "))
    if raw_lower.startswith("d") and not label.startswith("DA-OL") and norm not in {"ef"}:
        label = f"DA-OL {label}"
    if raw_lower.startswith("abs_"):
        label = f"abs {label}"
    if raw_lower.endswith("_anom"):
        label = f"{label} anomaly"
    if include_units:
        unit = metric_units(raw, context=context)
        if unit:
            label = f"{label} ({unit})"
    return label


def axis_label(metrics, fallback="value", context=None):
    metrics = [m for m in metrics if m]
    units = []
    for metric in metrics:
        unit = metric_units(metric, context=context)
        if unit and unit not in units:
            units.append(unit)
    if not units:
        return fallback
    if len(units) == 1:
        return f"{fallback} ({units[0]})"
    return f"{fallback} ({' / '.join(units)})"


def seconds_per_month(time_coord):
    days = time_coord.dt.days_in_month.astype("float64")
    return days * 86400.0


def select_season(da, season_year: int, season: str):
    months = SEASON_WINDOWS[season]
    mask = xr.zeros_like(da["time"].dt.month, dtype=bool)
    for month in months:
        year_for_month = season_year - 1 if (season == "DJF" and month == 12) else season_year
        mask = mask | ((da.time.dt.year == year_for_month) & (da.time.dt.month == month))
    return da.sel(time=mask)


def sum_preserve_nan(da, dim="time"):
    count = da.notnull().sum(dim)
    out = da.fillna(0.0).sum(dim)
    return out.where(count > 0)


def weighted_time_mean(da):
    weights = da.time.dt.days_in_month.astype("float64")
    return da.weighted(weights).mean("time", skipna=True)


def seasonal_aggregate(da, years, season: str, method: str, strict=True, name=None):
    pieces = []
    valid_years = []
    for year in years:
        sub = select_season(da, year, season)
        expected = len(SEASON_WINDOWS[season])
        if strict and sub.sizes.get("time", 0) != expected:
            warnings.warn(f"Skipping {season} {year}: expected {expected} months, found {sub.sizes.get('time', 0)}")
            continue
        if sub.sizes.get("time", 0) == 0:
            continue
        if method == "mean":
            out = weighted_time_mean(sub)
        elif method == "sum":
            out = sum_preserve_nan(sub, dim="time")
        elif method == "flux_total":
            out = sum_preserve_nan(sub * seconds_per_month(sub.time), dim="time")
        else:
            raise ValueError(f"Unknown aggregation method {method}")
        pieces.append(out)
        valid_years.append(year)
    if not pieces:
        raise ValueError(f"No valid {season} aggregates for years={years}")
    out = xr.concat(pieces, dim=pd.Index(valid_years, name="year"))
    if name:
        out.name = name
    return out


def get_model_var(experiment: str, var: str):
    if var not in VAR_SOURCE:
        raise KeyError(f"{var} is not available in matched OL/DA monthly files")
    return DATASETS[experiment][VAR_SOURCE[var]][var]


def seasonal_model_var(experiment: str, var: str, years, season: str):
    da = get_model_var(experiment, var)
    if var in WATER_FLUX_VARS:
        method = "flux_total"
    else:
        method = "mean"
    out = seasonal_aggregate(da, years, season, method=method, name=f"{experiment}_{var}_{season}")
    if var in WATER_FLUX_VARS:
        out.attrs["units"] = "kg m-2 season-1"
    return out


def seasonal_delta_var(var: str, years, season: str):
    out = seasonal_model_var("da", var, years, season) - seasonal_model_var("ol", var, years, season)
    out.name = f"d{var}_{season}"
    return out


def seasonal_catch_var(var: str, years, season: str):
    if var not in CATCH:
        raise KeyError(f"{var} missing from cumulative catch increment product")
    method = "sum"
    out = seasonal_aggregate(CATCH[var], years, season, method=method, name=f"{var}_{season}")
    return out


def seasonal_inst3_var(var: str, years, season: str):
    if var not in INST3:
        raise KeyError(f"{var} missing from raw inst3 diagnostic product")
    out = seasonal_aggregate(INST3[var], years, season, method="mean", name=f"{var}_{season}")
    return out


def mask_small_denominator(num, den, threshold, ratio_name):
    out = num / den.where(np.abs(den) >= threshold)
    out.name = ratio_name
    return out


def seasonal_model_derived(experiment: str, name: str, years, season: str):
    if name == "TOTAL_RUNOFF":
        return seasonal_model_var(experiment, "RUNSURFLAND", years, season) + seasonal_model_var(experiment, "BASEFLOWLAND", years, season)
    if name == "RUNOFF_RATIO":
        runoff = seasonal_model_derived(experiment, "TOTAL_RUNOFF", years, season)
        precip = seasonal_model_var(experiment, "PRECTOTCORRLAND", years, season)
        return mask_small_denominator(runoff, precip, 10.0, f"{experiment}_{name}_{season}")
    if name == "BASEFLOW_FRACTION":
        base = seasonal_model_var(experiment, "BASEFLOWLAND", years, season)
        runoff = seasonal_model_derived(experiment, "TOTAL_RUNOFF", years, season)
        return mask_small_denominator(base, runoff, 1.0, f"{experiment}_{name}_{season}")
    if name == "MELT_INFIL_FRACTION":
        qinfil = seasonal_model_var(experiment, "QINFILLAND", years, season)
        runsurf = seasonal_model_var(experiment, "RUNSURFLAND", years, season)
        return mask_small_denominator(qinfil, qinfil + runsurf, 1.0, f"{experiment}_{name}_{season}")
    if name == "ET_OVER_P":
        et = seasonal_model_var(experiment, "EVLAND", years, season)
        precip = seasonal_model_var(experiment, "PRECTOTCORRLAND", years, season)
        return mask_small_denominator(et, precip, 10.0, f"{experiment}_{name}_{season}")
    if name == "EF":
        le = seasonal_model_var(experiment, "LHLAND", years, season)
        h = seasonal_model_var(experiment, "SHLAND", years, season)
        return mask_small_denominator(le, le + h, 20.0, f"{experiment}_{name}_{season}")
    if name == "BOWEN":
        le = seasonal_model_var(experiment, "LHLAND", years, season)
        h = seasonal_model_var(experiment, "SHLAND", years, season)
        return mask_small_denominator(h, le, 10.0, f"{experiment}_{name}_{season}")
    component_map = {
        "TRANS_FRACTION": "LHLANDTRNS",
        "SOIL_EVAP_FRACTION": "LHLANDSOIL",
        "INTR_FRACTION": "LHLANDINTR",
        "SBLN_FRACTION": "LHLANDSBLN",
    }
    if name in component_map:
        comp = seasonal_model_var(experiment, component_map[name], years, season)
        le = seasonal_model_var(experiment, "LHLAND", years, season)
        return mask_small_denominator(comp, le, 10.0, f"{experiment}_{name}_{season}")
    raise KeyError(f"Unknown derived variable {name}")


def seasonal_response(name: str, years, season: str):
    if name in DERIVED_VARS:
        out = seasonal_model_derived("da", name, years, season) - seasonal_model_derived("ol", name, years, season)
    else:
        out = seasonal_delta_var(name, years, season)
    out.name = f"d{name}_{season}"
    return out


def monthly_model_var(experiment: str, var: str):
    da = get_model_var(experiment, var)
    if var in WATER_FLUX_VARS:
        out = da * seconds_per_month(da.time)
        out.attrs["units"] = "kg m-2 month-1"
        return out
    return da


def monthly_model_derived(experiment: str, name: str):
    if name == "TOTAL_RUNOFF":
        return monthly_model_var(experiment, "RUNSURFLAND") + monthly_model_var(experiment, "BASEFLOWLAND")
    if name == "EF":
        le = monthly_model_var(experiment, "LHLAND")
        h = monthly_model_var(experiment, "SHLAND")
        return mask_small_denominator(le, le + h, 20.0, f"{experiment}_{name}")
    raise KeyError(f"Monthly derived variable not implemented: {name}")


def monthly_response(name: str):
    if name in {"TOTAL_RUNOFF", "EF"}:
        return monthly_model_derived("da", name) - monthly_model_derived("ol", name)
    return monthly_model_var("da", name) - monthly_model_var("ol", name)


def broadcast_weights_like(da, mask=None):
    w = tile_weight.broadcast_like(da)
    if mask is not None:
        w = w.where(mask)
    return w.where(np.isfinite(w) & (w > 0))


def weighted_tile_mean(da, mask=None):
    if mask is not None:
        da = da.where(mask)
    w = broadcast_weights_like(da, mask=mask)
    try:
        return da.weighted(w).mean("tile", skipna=True)
    except Exception:
        return da.where(mask).mean("tile", skipna=True) if mask is not None else da.mean("tile", skipna=True)


def table_from_year_arrays(years, arrays: dict, mask, period=None) -> pd.DataFrame:
    tile_ids = np.arange(n_tile)
    frames = []
    mask_is_yearly = hasattr(mask, "dims") and "year" in mask.dims
    for year in years:
        if mask_is_yearly:
            valid = np.asarray(mask.sel(year=year).values, dtype=bool)
        else:
            valid = np.asarray(mask.values if hasattr(mask, "values") else mask, dtype=bool)
        rec = {
            "year": np.full(valid.sum(), year, dtype=int),
            "tile": tile_ids[valid],
            "lat": lat.values[valid],
            "lon": lon.values[valid],
            "area": tile_weight.values[valid],
        }
        for name, arr in arrays.items():
            values = arr.sel(year=year).values if "year" in arr.dims else arr.values
            rec[name] = np.asarray(values)[valid]
        frame = pd.DataFrame(rec)
        if period is not None:
            frame["period"] = period
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def binned_summary(df, x_col, y_col, by_cols=None, n_bins=8, min_count=20, bin_kind="quantile"):
    by_cols = by_cols or []
    frames = []
    group_iter = [((), df)] if not by_cols else df.groupby(by_cols, dropna=False)
    for key, sub in group_iter:
        keep = [x_col, y_col] + by_cols
        sub = sub[keep].replace([np.inf, -np.inf], np.nan).dropna(subset=[x_col, y_col])
        if sub.empty or sub[x_col].nunique() < 2:
            continue
        try:
            bins = pd.qcut(sub[x_col], q=n_bins, duplicates="drop")
        except ValueError:
            continue
        grouped = sub.assign(bin=bins).groupby("bin", observed=True)
        out = grouped.agg(
            n=(y_col, "size"),
            x_mean=(x_col, "mean"),
            x_median=(x_col, "median"),
            x_min=(x_col, "min"),
            x_max=(x_col, "max"),
            y_mean=(y_col, "mean"),
            y_median=(y_col, "median"),
            y_q25=(y_col, lambda x: np.nanpercentile(x, 25)),
            y_q75=(y_col, lambda x: np.nanpercentile(x, 75)),
            y_std=(y_col, "std"),
        ).reset_index()
        out = out[out["n"] >= min_count].copy()
        if out.empty:
            continue
        out["y_iqr"] = out["y_q75"] - out["y_q25"]
        out["y_se"] = out["y_std"] / np.sqrt(out["n"])
        out["bin_label"] = out["bin"].astype(str)
        out = out.drop(columns=["bin"])
        if by_cols:
            if not isinstance(key, tuple):
                key = (key,)
            for col, val in zip(by_cols, key):
                out[col] = val
        out["x_metric"] = x_col
        out["y_metric"] = y_col
        out["bin_kind"] = bin_kind
        frames.append(out)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def add_within_tile_anomalies(df, cols, group_cols=("period", "tile")):
    out = df.copy()
    present_group_cols = [c for c in group_cols if c in out.columns]
    for col in cols:
        if col in out.columns:
            out[f"{col}_anom"] = out[col] - out.groupby(present_group_cols)[col].transform("mean")
    return out


def high_low_signal(summary, x_metric, y_metric, period=None):
    if summary.empty:
        return np.nan
    key = summary[(summary["x_metric"] == x_metric) & (summary["y_metric"] == y_metric)].copy()
    if period is not None and "period" in key:
        key = key[key["period"] == period]
    if key.empty:
        return np.nan
    ordered = key.sort_values("x_mean")
    return float(ordered.iloc[-1]["y_mean"] - ordered.iloc[0]["y_mean"])


def corr_by_tile(x, y, min_years=6):
    x, y = xr.align(x, y, join="inner")
    valid_count = (x.notnull() & y.notnull()).sum("year")
    corr = xr.corr(x, y, dim="year")
    return corr.where(valid_count >= min_years), valid_count


def plot_binned_lines(
    summary,
    x_metric,
    y_metrics,
    by_col=None,
    title="",
    xlabel="",
    ylabel="",
    stem=None,
    secondary_y_metrics=None,
    secondary_ylabel="",
    unit_context="seasonal",
):
    if summary.empty:
        print("No binned data for", title)
        return
    secondary_y_metrics = set(secondary_y_metrics or [])
    primary_metrics = [m for m in y_metrics if m not in secondary_y_metrics]
    secondary_metrics = [m for m in y_metrics if m in secondary_y_metrics]
    if not primary_metrics and secondary_metrics:
        primary_metrics, secondary_metrics = secondary_metrics, []
    if by_col is None:
        fig, ax = plt.subplots(figsize=(7.2, 4.4))
        axes = [ax]
        groups = [(None, summary)]
    else:
        vals = list(summary[by_col].dropna().unique())
        fig, axes = plt.subplots(1, len(vals), figsize=(5.6 * len(vals), 4.4), sharey=(len(secondary_metrics) == 0))
        if len(vals) == 1:
            axes = [axes]
        groups = [(val, summary[summary[by_col] == val]) for val in vals]

    for ax, (group_val, group_df) in zip(axes, groups):
        twin = ax.twinx() if secondary_metrics else None
        for metric in primary_metrics:
            sub = group_df[(group_df["x_metric"] == x_metric) & (group_df["y_metric"] == metric)].sort_values("x_mean")
            if sub.empty:
                continue
            ax.errorbar(sub["x_mean"], sub["y_mean"], yerr=sub["y_se"], marker="o", linewidth=1.5, capsize=2, label=metric_label(metric, include_units=True, context=unit_context))
        if twin is not None:
            for metric in secondary_metrics:
                sub = group_df[(group_df["x_metric"] == x_metric) & (group_df["y_metric"] == metric)].sort_values("x_mean")
                if sub.empty:
                    continue
                twin.errorbar(sub["x_mean"], sub["y_mean"], yerr=sub["y_se"], marker="s", linestyle="--", linewidth=1.3, capsize=2, label=metric_label(metric, include_units=True, context=unit_context))
            twin.axhline(0, color="0.45", linewidth=0.6, linestyle=":")
            twin.set_ylabel(secondary_ylabel or axis_label(secondary_metrics, fallback="secondary response", context=unit_context))
        ax.axhline(0, color="0.2", linewidth=0.8)
        ax.set_xlabel(xlabel or metric_label(x_metric, include_units=True, context=unit_context))
        ax.set_ylabel(ylabel or axis_label(primary_metrics, fallback="response", context=unit_context))
        ax.grid(True, alpha=0.25)
        if group_val is not None:
            label = PERIODS.get(group_val, {}).get("label", str(group_val))
            ax.set_title(label)
        handles, labels = ax.get_legend_handles_labels()
        if twin is not None:
            h2, l2 = twin.get_legend_handles_labels()
            handles += h2
            labels += l2
        if handles:
            ax.legend(handles, labels, fontsize=7, loc="best")
    fig.suptitle(title)
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    if stem:
        savefig(fig, stem)
    show_figure(fig)


def plot_map_grid(map_items, mask, title, stem, extent=None, ncols=4):
    projection = ccrs.Robinson() if HAS_CARTOPY else None
    n = len(map_items)
    nrows = int(np.ceil(n / ncols))
    fig = plt.figure(figsize=(3.8 * ncols, 2.55 * nrows))
    for i, item in enumerate(map_items, 1):
        values = item["values"]
        cmap = item.get("cmap", "RdBu_r")
        norm = item.get("norm")
        if norm is None:
            norm = positive_norm(values.where(mask).values) if item.get("positive", False) else symmetric_norm(values.where(mask).values)
        ax = fig.add_subplot(nrows, ncols, i, projection=projection) if HAS_CARTOPY else fig.add_subplot(nrows, ncols, i)
        im = tile_scatter_map(ax, values.values, mask=mask.values if hasattr(mask, "values") else mask, title=item["title"], cmap=cmap, norm=norm, s=item.get("s", 0.65), extent=extent)
        cb = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.055, pad=0.04)
        cb.ax.tick_params(labelsize=7)
        unit_label = item.get("units") or metric_units(item.get("metric") or getattr(values, "name", "") or item.get("title", ""), context="seasonal")
        if unit_label:
            cb.set_label(unit_label, fontsize=7)
    fig.suptitle(title, fontsize=12)
    savefig(fig, stem)
    show_figure(fig)


## Open Monthly Data and Build Masks

The seasonal-snow mask is Northern Hemisphere focused and excludes persistent/permanent snow. A warm snow-free static mask is used for domain time series; Analysis C also uses year-specific warm snow-free masks.


### Figure Notes: Mask and Snow-Activity Context

The next cell creates `monthly_synthesis_masks_and_snow_activity_context`. It is a four-panel tile-space map used to document the sampling domains. All panels are plotted over valid land tiles.

- `Max OL/DA snow-cover fraction` maps the maximum monthly `FRLANDSNO` reached in either OL or DA. It shows where the model ever carries snow cover.
- `Max OL/DA snow mass` maps the maximum monthly `SNOMASLAND` reached in either OL or DA, in `kg m-2`. Together with `FRLANDSNO`, this defines snow-possible regions.
- `MAM snow DA activity, 2001-2023` maps the 2001-2023 mean MAM `snow_abs_netpack`. For each year this is `sum(abs(WESNN1_INCR + WESNN2_INCR + WESNN3_INCR))`, in `kg m-2`, so it measures snow-DA correction activity regardless of sign.
- `Static masks: warm=1, snow=2` shows the broad warm mostly snow-free mask and the NH seasonal-snow mask. The NH seasonal-snow mask requires latitude > 20 degrees N, snow possible in either OL or DA (`max FRLANDSNO > 0.05` or `max SNOMASLAND > 5 kg m-2`), and mean JJA `FRLANDSNO < 0.20` to exclude persistent snow/ice.


In [ ]:
scf_pair_max = xr.concat([get_model_var("ol", "FRLANDSNO"), get_model_var("da", "FRLANDSNO")], dim="experiment").max("experiment")
swe_pair_max = xr.concat([get_model_var("ol", "SNOMASLAND"), get_model_var("da", "SNOMASLAND")], dim="experiment").max("experiment")
scf_any = scf_pair_max.max("time", skipna=True).load()
swe_any = swe_pair_max.max("time", skipna=True).load()
jja_scf = scf_pair_max.sel(time=scf_pair_max.time.dt.month.isin([6, 7, 8])).mean("time", skipna=True).load()
valid_land_mask = (np.isfinite(lat) & np.isfinite(lon) & np.isfinite(tile_weight)).load()
seasonal_snow_mask = ((lat > NH_MIN_LAT) & ((scf_any > SNOW_POSSIBLE_SCF_THRESHOLD) | (swe_any > SNOW_POSSIBLE_SWE_THRESHOLD)) & (jja_scf < PERMANENT_SNOW_JJA_MEAN_MAX) & valid_land_mask).load()
snow_possible_mask = ((lat > NH_MIN_LAT) & ((scf_any > SNOW_POSSIBLE_SCF_THRESHOLD) | (swe_any > SNOW_POSSIBLE_SWE_THRESHOLD)) & valid_land_mask).load()
warm_static_mask = ((np.abs(lat) < 60.0) & (scf_any < WARM_SNOWFREE_SCF_MAX) & valid_land_mask).load()

high_years = list(range(2001, 2024))
high_mam_snow_abs = seasonal_catch_var("snow_abs_netpack", high_years, "MAM").mean("year", skipna=True).load()
high_threshold = float(np.nanpercentile(high_mam_snow_abs.where(seasonal_snow_mask).values, 75))
high_snow_activity_mask = ((seasonal_snow_mask) & (high_mam_snow_abs >= high_threshold)).load()

mask_summary = pd.DataFrame([
    {"mask": "valid land", "n_tiles": int(valid_land_mask.sum()), "area_weighted": use_area_weights, "definition": "finite lat/lon and positive tile area if available"},
    {"mask": "NH snow possible", "n_tiles": int(snow_possible_mask.sum()), "area_weighted": use_area_weights, "definition": f"lat>{NH_MIN_LAT} and max(OL/DA FRLANDSNO>{SNOW_POSSIBLE_SCF_THRESHOLD} or SNOMASLAND>{SNOW_POSSIBLE_SWE_THRESHOLD})"},
    {"mask": "NH seasonal snow", "n_tiles": int(seasonal_snow_mask.sum()), "area_weighted": use_area_weights, "definition": f"snow possible and mean JJA FRLANDSNO<{PERMANENT_SNOW_JJA_MEAN_MAX}"},
    {"mask": "high snow-DA activity", "n_tiles": int(high_snow_activity_mask.sum()), "area_weighted": use_area_weights, "definition": f"NH seasonal snow and 2001-2023 MAM mean snow_abs_netpack >= upper quartile ({high_threshold:.3g} kg m-2)"},
    {"mask": "warm mostly snow-free static", "n_tiles": int(warm_static_mask.sum()), "area_weighted": use_area_weights, "definition": f"abs(lat)<60 and max monthly OL/DA FRLANDSNO<{WARM_SNOWFREE_SCF_MAX}"},
])
mask_summary.to_csv(OUT_DIR / "monthly_synthesis_mask_summary.csv", index=False)
display(mask_summary)

mask_map_values = xr.where(seasonal_snow_mask, 2, xr.where(warm_static_mask, 1, np.nan)).load()
plot_map_grid(
    [
        {"values": scf_any, "title": "Max OL/DA snow-cover fraction", "cmap": "viridis", "positive": True},
        {"values": swe_any, "title": "Max OL/DA snow mass", "cmap": "viridis", "positive": True},
        {"values": high_mam_snow_abs, "title": "MAM snow DA activity, 2001-2023", "cmap": "magma", "positive": True},
        {"values": mask_map_values, "title": "Static masks: warm=1, snow=2", "cmap": "viridis", "positive": True},
    ],
    mask=valid_land_mask,
    title="Monthly synthesis masks and snow-activity context",
    stem="monthly_synthesis_masks_and_snow_activity_context",
    extent=[-180, 180, -60, 90],
    ncols=2,
)


def warm_snowfree_season_mask(years, season):
    fr_ol = seasonal_model_var("ol", "FRLANDSNO", years, season)
    fr_da = seasonal_model_var("da", "FRLANDSNO", years, season)
    tsoil_ol = seasonal_model_var("ol", "TSOIL1", years, season)
    tsoil_da = seasonal_model_var("da", "TSOIL1", years, season)
    mask = (
        (fr_ol < WARM_SNOWFREE_SCF_MAX)
        & (fr_da < WARM_SNOWFREE_SCF_MAX)
        & (tsoil_ol > WARM_SNOWFREE_TSOIL_MIN_K)
        & (tsoil_da > WARM_SNOWFREE_TSOIL_MIN_K)
        & valid_land_mask
    )
    return mask.load()


## Analysis A: MODIS-Only Snow DA Carryover Into Soil Moisture and Hydrology

This is the cleanest causal diagnostic because season years 2001-2006 precede microwave soil-moisture DA. Predictors are raw cumulative MAM snow increments; responses are AMJ/MJJ/JJA `DA - OL` states, water totals, and energy means.


### Figure Notes: Analysis A Process Maps and Binned Relationships

The next cell creates three Analysis A figures from NH seasonal-snow tile-years in the MODIS-only clean years 2001-2006, before microwave soil-moisture DA is active. The tile-year table includes MAM snow increments and AMJ/MJJ/JJA DA-OL responses.

`analysisA_process_chain_maps` maps clean-year means. `MAM snow_net` is the signed cumulative MAM snow-mass increment, `sum(WESNN1_INCR + WESNN2_INCR + WESNN3_INCR)`, in `kg m-2`; positive means snow DA added SWE and negative means it removed SWE. `MAM snow_abs_netpack` is the cumulative absolute snow correction, `sum(abs(WESNN1_INCR + WESNN2_INCR + WESNN3_INCR))`, in `kg m-2`. The response panels show `AMJ DA-OL snowmelt` (`dSMLAND_amj`), `AMJ DA-OL infiltration` (`dQINFILLAND_amj`), `MJJ DA-OL RZMC`, `MJJ DA-OL ET`, `MJJ DA-OL total runoff`, and `MJJ DA-OL total water`. Fluxes are converted to seasonal totals (`kg m-2 season-1`); `RZMC` is `m3 m-3`; `TWLAND` is storage in `kg m-2`.

`analysisA_binned_signed_snow_to_hydrology` asks whether signed MAM snow correction predicts signed later hydrologic response. The x-axis is `snow_net_mam` in `kg m-2`, split into 5 quantile bins. Points are bin means and error bars are standard errors. The primary y-axis shows water/storage response variables such as `dQINFILLAND_amj`, `dEVLAND_mjj`, `dTOTAL_RUNOFF_mjj`, and `dTWLAND_mjj`. The secondary y-axis shows `dRZMC_mjj` in `m3 m-3` because its scale is much smaller.

`analysisA_binned_snow_activity_to_response_magnitude` asks whether stronger snow-DA activity is associated with larger response magnitude. The x-axis is `snow_abs_netpack_mam` in `kg m-2`, split into 8 quantile bins. The y variables are absolute response magnitudes such as `abs_dEVLAND_mjj`, `abs_dTOTAL_RUNOFF_mjj`, `abs_dTWLAND_mjj`, and `abs_dRZMC_mjj`. This removes sign and should be read as activity versus response size, not as independent validation.


In [ ]:
years_A = PERIODS["modis_only"]["clean_years"]

A_arrays = {
    "snow_net_mam": seasonal_catch_var("snow_net", years_A, "MAM").load(),
    "snow_abs_netpack_mam": seasonal_catch_var("snow_abs_netpack", years_A, "MAM").load(),
    "snow_event_count_mam": seasonal_catch_var("snow_event_count", years_A, "MAM").load(),
    "snow_abs_layers_mam": seasonal_catch_var("snow_abs_layers", years_A, "MAM").load(),
    "dFRLANDSNO_mam": seasonal_response("FRLANDSNO", years_A, "MAM").load(),
    "dSNOMASLAND_mam": seasonal_response("SNOMASLAND", years_A, "MAM").load(),
}

A_response_specs = [
    ("AMJ", ["SMLAND", "QINFILLAND", "FRLANDSNO", "SNOMASLAND", "SFMC", "RZMC", "TWLAND", "WCHANGELAND", "EVLAND", "RUNSURFLAND", "BASEFLOWLAND", "TOTAL_RUNOFF", "BASEFLOW_FRACTION", "MELT_INFIL_FRACTION", "LHLAND", "SHLAND", "EF", "GHLAND", "TSOIL1"]),
    ("MJJ", ["SMLAND", "QINFILLAND", "FRLANDSNO", "SNOMASLAND", "SFMC", "RZMC", "TWLAND", "WCHANGELAND", "EVLAND", "RUNSURFLAND", "BASEFLOWLAND", "TOTAL_RUNOFF", "BASEFLOW_FRACTION", "MELT_INFIL_FRACTION", "LHLAND", "SHLAND", "EF", "GHLAND", "TSOIL1"]),
    ("JJA", ["SFMC", "RZMC", "TWLAND", "EVLAND", "TOTAL_RUNOFF", "LHLAND", "SHLAND", "EF", "TSOIL1"]),
]
for season, variables in A_response_specs:
    for var in variables:
        try:
            A_arrays[f"d{var.lower()}_{season.lower()}"] = seasonal_response(var, years_A, season).load()
        except Exception as exc:
            warnings.warn(f"Analysis A skipped {var} {season}: {exc}")

A_table = table_from_year_arrays(years_A, A_arrays, seasonal_snow_mask, period="modis_only")
A_sample_summary = pd.DataFrame([{
    "analysis": "A",
    "period": "modis_only",
    "n_rows": len(A_table),
    "n_years": len(years_A),
    "n_tiles_static_mask": int(seasonal_snow_mask.sum()),
    "domain": "NH seasonal snow",
}])
A_sample_summary.to_csv(OUT_DIR / "analysisA_sample_summary.csv", index=False)
display(A_sample_summary)
display(A_table.head())

A_signed_targets = [
    "dsmland_amj", "dqinfilland_amj", "drzmc_mjj", "devland_mjj", "dtotal_runoff_mjj", "dtwland_mjj",
    "dlhland_mjj", "dshland_mjj", "def_mjj",
]
A_signed_targets = [c for c in A_signed_targets if c in A_table.columns]
A_activity_targets = []
for c in A_signed_targets:
    A_table[f"abs_{c}"] = np.abs(A_table[c])
    A_activity_targets.append(f"abs_{c}")

A_signed_bins = []
for target in A_signed_targets:
    out = binned_summary(A_table, "snow_net_mam", target, n_bins=5, bin_kind="signed_snow_net")
    if not out.empty:
        A_signed_bins.append(out)
analysisA_binned_signed = pd.concat(A_signed_bins, ignore_index=True) if A_signed_bins else pd.DataFrame()
analysisA_binned_signed.to_csv(OUT_DIR / "analysisA_binned_signed_snow_to_hydrology.csv", index=False)
display(analysisA_binned_signed.head(30))

A_activity_bins = []
for target in A_activity_targets:
    out = binned_summary(A_table, "snow_abs_netpack_mam", target, n_bins=N_BINS, bin_kind="snow_activity")
    if not out.empty:
        A_activity_bins.append(out)
analysisA_binned_activity = pd.concat(A_activity_bins, ignore_index=True) if A_activity_bins else pd.DataFrame()
analysisA_binned_activity.to_csv(OUT_DIR / "analysisA_binned_snow_activity_to_response_magnitude.csv", index=False)
display(analysisA_binned_activity.head(30))

A_map_items = [
    {"values": A_arrays["snow_net_mam"].mean("year"), "title": "MAM snow_net", "metric": "snow_net_mam", "cmap": "RdBu_r"},
    {"values": A_arrays["snow_abs_netpack_mam"].mean("year"), "title": "MAM snow_abs_netpack", "metric": "snow_abs_netpack_mam", "cmap": "magma", "positive": True},
    {"values": A_arrays["dsmland_amj"].mean("year"), "title": "AMJ DA-OL snowmelt", "metric": "dsmland_amj", "cmap": "RdBu_r"},
    {"values": A_arrays["dqinfilland_amj"].mean("year"), "title": "AMJ DA-OL infiltration", "metric": "dqinfilland_amj", "cmap": "RdBu_r"},
    {"values": A_arrays["drzmc_mjj"].mean("year"), "title": "MJJ DA-OL RZMC", "metric": "drzmc_mjj", "cmap": "RdBu_r"},
    {"values": A_arrays["devland_mjj"].mean("year"), "title": "MJJ DA-OL ET", "metric": "devland_mjj", "cmap": "RdBu_r"},
    {"values": A_arrays["dtotal_runoff_mjj"].mean("year"), "title": "MJJ DA-OL total runoff", "metric": "dtotal_runoff_mjj", "cmap": "RdBu_r"},
    {"values": A_arrays["dtwland_mjj"].mean("year"), "title": "MJJ DA-OL total water", "metric": "dtwland_mjj", "cmap": "RdBu_r"},
]
plot_map_grid(A_map_items, seasonal_snow_mask, "Analysis A: MODIS-only snow DA carryover, 2001-2006", "analysisA_process_chain_maps", extent=[-180, 180, 20, 90], ncols=4)

plot_binned_lines(
    analysisA_binned_signed,
    x_metric="snow_net_mam",
    y_metrics=[m for m in ["drzmc_mjj", "dqinfilland_amj", "devland_mjj", "dtotal_runoff_mjj", "dtwland_mjj"] if m in A_signed_targets],
    title="Analysis A: signed MAM snow correction vs later hydrologic response",
    xlabel="MAM cumulative snow_net (kg m-2)",
    ylabel="Water response (kg m-2; fluxes are seasonal totals)",
    secondary_y_metrics=["drzmc_mjj"],
    secondary_ylabel="RZMC response (m3 m-3)",
    stem="analysisA_binned_signed_snow_to_hydrology",
)
plot_binned_lines(
    analysisA_binned_activity,
    x_metric="snow_abs_netpack_mam",
    y_metrics=[m for m in ["abs_drzmc_mjj", "abs_devland_mjj", "abs_dtotal_runoff_mjj", "abs_dtwland_mjj"] if m in A_activity_targets],
    title="Analysis A: MAM snow DA activity vs response magnitude",
    xlabel="MAM snow_abs_netpack (kg m-2)",
    ylabel="Absolute water response (kg m-2; fluxes are seasonal totals)",
    secondary_y_metrics=["abs_drzmc_mjj"],
    secondary_ylabel="Absolute RZMC response (m3 m-3)",
    stem="analysisA_binned_snow_activity_to_response_magnitude",
)


### Figure Notes: Analysis A Monthly Composites by Snow-Net Group

The next cell creates `analysisA_monthly_composite_by_snow_net_group`. It uses the same 2001-2006 NH seasonal-snow tile-years as Analysis A. Each tile-year is grouped by its MAM `snow_net`: lower quartile (`snow removal lower quartile`), upper quartile (`snow addition upper quartile`), and the middle 50 percent (`near-zero / middle 50%`).

For each group, variable, and calendar month January-August, the plotted value is an area-weighted mean DA-OL response across the selected tile-years. Panels show `SNOMASLAND`, `FRLANDSNO`, `SMLAND`, `QINFILLAND`, `RZMC`, `EVLAND`, `TOTAL_RUNOFF`, `TWLAND`, `LHLAND`, and `SHLAND`. Flux variables are monthly totals (`kg m-2 month-1`), states keep native units, and turbulent heat fluxes are `W m-2`. The figure is mainly about timing: when snow-addition and snow-removal cases separate and how the response carries into melt, soil moisture, runoff, storage, and energy fluxes.


In [ ]:
# Composite Jan-Aug monthly evolution by MAM snow_net group.
A_group_basis = A_table[["year", "tile", "snow_net_mam"]].replace([np.inf, -np.inf], np.nan).dropna()
q25, q75 = A_group_basis["snow_net_mam"].quantile([0.25, 0.75]).values
A_group_basis["snow_net_group"] = "near-zero / middle 50%"
A_group_basis.loc[A_group_basis["snow_net_mam"] <= q25, "snow_net_group"] = "snow removal lower quartile"
A_group_basis.loc[A_group_basis["snow_net_mam"] >= q75, "snow_net_group"] = "snow addition upper quartile"

def monthly_values_for_response(name):
    return monthly_response(name)

composite_vars = ["SNOMASLAND", "FRLANDSNO", "SMLAND", "QINFILLAND", "SFMC", "RZMC", "EVLAND", "TOTAL_RUNOFF", "TWLAND", "LHLAND", "SHLAND"]
composite_rows = []
for group_name, group_df in A_group_basis.groupby("snow_net_group"):
    group_by_year = {year: sub["tile"].to_numpy(dtype=int) for year, sub in group_df.groupby("year")}
    for var in composite_vars:
        try:
            da_monthly = monthly_values_for_response(var)
        except Exception as exc:
            warnings.warn(f"Composite skipped {var}: {exc}")
            continue
        for month in range(1, 9):
            numer = 0.0
            denom = 0.0
            for year, tiles in group_by_year.items():
                sub = da_monthly.sel(time=((da_monthly.time.dt.year == year) & (da_monthly.time.dt.month == month)))
                if sub.sizes.get("time", 0) != 1 or tiles.size == 0:
                    continue
                vals = np.asarray(sub.isel(time=0).values)[tiles]
                w = np.asarray(tile_weight.values)[tiles]
                ok = np.isfinite(vals) & np.isfinite(w) & (w > 0)
                if ok.any():
                    numer += float(np.nansum(vals[ok] * w[ok]))
                    denom += float(np.nansum(w[ok]))
            composite_rows.append({
                "analysis": "A",
                "group": group_name,
                "variable": var,
                "month": month,
                "value": numer / denom if denom > 0 else np.nan,
                "weighted": use_area_weights,
                "q25_snow_net": q25,
                "q75_snow_net": q75,
            })
analysisA_composite = pd.DataFrame(composite_rows)
analysisA_composite.to_csv(OUT_DIR / "analysisA_monthly_group_composite.csv", index=False)
display(analysisA_composite.head(30))

plot_vars = ["SNOMASLAND", "FRLANDSNO", "SMLAND", "QINFILLAND", "RZMC", "EVLAND", "TOTAL_RUNOFF", "TWLAND", "LHLAND", "SHLAND"]
fig, axes = plt.subplots(2, 5, figsize=(17, 6.2), sharex=True)
for ax, var in zip(axes.ravel(), plot_vars):
    sub = analysisA_composite[analysisA_composite["variable"] == var]
    for group_name, sg in sub.groupby("group"):
        ax.plot(sg["month"], sg["value"], marker="o", linewidth=1.3, label=group_name)
    ax.axhline(0, color="0.2", linewidth=0.7)
    unit = metric_units(var, context="monthly")
    title = metric_label(var)
    ax.set_title(f"{title}\n[{unit}]" if unit else title, fontsize=9)
    ax.set_ylabel("DA-OL")
    ax.grid(True, alpha=0.25)
for ax in axes[1, :]:
    ax.set_xlabel("month")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8)
fig.suptitle("Analysis A: Jan-Aug DA-OL composites grouped by MAM snow_net, 2001-2006")
fig.subplots_adjust(bottom=0.18, top=0.88)
savefig(fig, "analysisA_monthly_composite_by_snow_net_group")
show_figure(fig)


## Analysis B: Snow DA Activity and Later Soil-Moisture DA Activity

This analysis asks whether strong prior snow DA is associated with later soil-moisture analysis correction magnitude or direction in the microwave periods. Raw relationships are paired with within-tile anomaly controls to reduce geography-only interpretation.


### Figure Notes: Analysis B Binned Snow-DA Versus SM-DA Relationships

The next cell creates three Analysis B binned figures from NH seasonal-snow tile-years in the microwave periods: clean years 2008-2014 for the microwave pre-SMAP period and 2016-2023 for the SMAP-era microwave period.

`analysisB_binned_snow_activity_vs_sm_activity` asks whether prior snow-DA activity predicts later soil-moisture DA activity. The x-axis is MAM `snow_abs_netpack` in `kg m-2`, split into 8 quantile bins within each period. The primary y-axis shows `soil_water_abs_activity_mjj`, the seasonal absolute native prognostic soil-water increment activity in `kg m-2`. The secondary y-axis shows diagnostic volumetric activity metrics such as `RZMC_INC_RMS_mjj` and `SFMC_INC_RMS_mjj`, in `m3 m-3`; these are ANA-FCST RMS diagnostics, not cumulative water corrections.

`analysisB_binned_within_tile_anomaly_controls` repeats the activity-versus-activity plot after subtracting each tile's period-mean value from each variable. The x-axis is `snow_abs_netpack_mam_anom`. The y variables are within-tile anomalies of the activity metrics. This controls for fixed geography: a strong raw relationship that disappears here is probably mostly spatial climatology rather than year-to-year coupling.

`analysisB_binned_signed_snow_vs_signed_sm_correction` uses signed MAM `snow_net` in `kg m-2`, split into 5 quantile bins, to compare against signed later corrections. The secondary axis shows `RZMC_INC_MEAN_mjj` and `SFMC_INC_MEAN_mjj` in `m3 m-3`; positive means analysis wetter than forecast. The primary axis shows `soil_water_net_approx_mjj` in `kg m-2`, computed approximately as `SRFEXC_net + RZEXC_net - CATDEF_net` because `CATDEF` is a deficit. This signed soil-water metric is useful but should not be treated as exact water-budget closure.


In [ ]:
def build_analysisB_period(period_key, years):
    arrays = {
        "snow_abs_netpack_mam": seasonal_catch_var("snow_abs_netpack", years, "MAM").load(),
        "snow_net_mam": seasonal_catch_var("snow_net", years, "MAM").load(),
        "snow_event_count_mam": seasonal_catch_var("snow_event_count", years, "MAM").load(),
        "snow_abs_netpack_fma": seasonal_catch_var("snow_abs_netpack", years, "FMA").load(),
        "snow_net_fma": seasonal_catch_var("snow_net", years, "FMA").load(),
    }
    for season in ["AMJ", "MJJ", "JJA"]:
        for var in [
            "SFMC_INC_ABS_MEAN", "SFMC_INC_RMS", "SFMC_INC_MEAN",
            "RZMC_INC_ABS_MEAN", "RZMC_INC_RMS", "RZMC_INC_MEAN",
            "PRMC_INC_ABS_MEAN", "PRMC_INC_RMS", "PRMC_INC_MEAN",
            "TSURF_INC_RMS", "TSURF_INC_MEAN", "TSOIL1_INC_RMS", "TSOIL1_INC_MEAN",
        ]:
            if var in INST3:
                arrays[f"{var.lower()}_{season.lower()}"] = seasonal_inst3_var(var, years, season).load()
        for var in ["soil_water_abs_activity", "soil_water_net_approx", "srfexc_net", "rzexc_net", "catdef_net"]:
            if var in CATCH:
                arrays[f"{var}_{season.lower()}"] = seasonal_catch_var(var, years, season).load()
    table = table_from_year_arrays(years, arrays, seasonal_snow_mask, period=period_key)
    return table, arrays

B_tables = []
B_ARRAYS = {}
for period_key in ["pre_smap_mw", "smap_era"]:
    table, arrays = build_analysisB_period(period_key, PERIODS[period_key]["clean_years"])
    B_tables.append(table)
    B_ARRAYS[period_key] = arrays
analysisB_table = pd.concat(B_tables, ignore_index=True) if B_tables else pd.DataFrame()
analysisB_sample_summary = analysisB_table.groupby("period").size().reset_index(name="n_rows") if not analysisB_table.empty else pd.DataFrame()
analysisB_sample_summary["analysis"] = "B"
analysisB_sample_summary.to_csv(OUT_DIR / "analysisB_sample_summary.csv", index=False)
display(analysisB_sample_summary)
display(analysisB_table.head())

B_activity_targets = [
    "rzmc_inc_rms_amj", "rzmc_inc_rms_mjj", "rzmc_inc_abs_mean_mjj",
    "sfmc_inc_rms_mjj", "sfmc_inc_abs_mean_mjj", "prmc_inc_rms_mjj",
    "soil_water_abs_activity_mjj", "soil_water_abs_activity_jja",
]
B_activity_targets = [c for c in B_activity_targets if c in analysisB_table.columns]
B_signed_targets = [
    "rzmc_inc_mean_amj", "rzmc_inc_mean_mjj", "sfmc_inc_mean_mjj", "prmc_inc_mean_mjj", "soil_water_net_approx_mjj",
]
B_signed_targets = [c for c in B_signed_targets if c in analysisB_table.columns]

B_raw_bins = []
for target in B_activity_targets:
    out = binned_summary(analysisB_table, "snow_abs_netpack_mam", target, by_cols=["period"], n_bins=N_BINS, bin_kind="snow_activity")
    if not out.empty:
        B_raw_bins.append(out)
analysisB_binned_activity = pd.concat(B_raw_bins, ignore_index=True) if B_raw_bins else pd.DataFrame()
analysisB_binned_activity.to_csv(OUT_DIR / "analysisB_binned_snow_activity_vs_sm_activity.csv", index=False)
display(analysisB_binned_activity.head(30))

B_signed_bins = []
for target in B_signed_targets:
    out = binned_summary(analysisB_table, "snow_net_mam", target, by_cols=["period"], n_bins=5, bin_kind="signed_snow_net")
    if not out.empty:
        B_signed_bins.append(out)
analysisB_binned_signed = pd.concat(B_signed_bins, ignore_index=True) if B_signed_bins else pd.DataFrame()
analysisB_binned_signed.to_csv(OUT_DIR / "analysisB_binned_signed_snow_vs_signed_sm_correction.csv", index=False)
display(analysisB_binned_signed.head(30))

anom_cols = ["snow_abs_netpack_mam", "snow_net_mam"] + B_activity_targets + B_signed_targets
analysisB_anom_table = add_within_tile_anomalies(analysisB_table, anom_cols, group_cols=("period", "tile"))
B_anom_bins = []
for target in [f"{c}_anom" for c in B_activity_targets if f"{c}_anom" in analysisB_anom_table.columns]:
    out = binned_summary(analysisB_anom_table, "snow_abs_netpack_mam_anom", target, by_cols=["period"], n_bins=N_BINS, bin_kind="within_tile_snow_activity_anomaly")
    if not out.empty:
        B_anom_bins.append(out)
for target in [f"{c}_anom" for c in B_signed_targets if f"{c}_anom" in analysisB_anom_table.columns]:
    out = binned_summary(analysisB_anom_table, "snow_net_mam_anom", target, by_cols=["period"], n_bins=5, bin_kind="within_tile_signed_snow_anomaly")
    if not out.empty:
        B_anom_bins.append(out)
analysisB_binned_anomaly = pd.concat(B_anom_bins, ignore_index=True) if B_anom_bins else pd.DataFrame()
analysisB_binned_anomaly.to_csv(OUT_DIR / "analysisB_binned_within_tile_anomaly_controls.csv", index=False)
display(analysisB_binned_anomaly.head(30))

plot_binned_lines(
    analysisB_binned_activity,
    x_metric="snow_abs_netpack_mam",
    y_metrics=[m for m in ["rzmc_inc_rms_mjj", "sfmc_inc_rms_mjj", "soil_water_abs_activity_mjj"] if m in B_activity_targets],
    by_col="period",
    title="Analysis B: MAM snow DA activity vs later SM DA activity",
    xlabel="MAM snow_abs_netpack (kg m-2)",
    ylabel="Native water-increment activity (kg m-2 season-1)",
    secondary_y_metrics=["rzmc_inc_rms_mjj", "sfmc_inc_rms_mjj"],
    secondary_ylabel="Diagnostic SM activity (m3 m-3)",
    stem="analysisB_binned_snow_activity_vs_sm_activity",
)
plot_binned_lines(
    analysisB_binned_anomaly,
    x_metric="snow_abs_netpack_mam_anom",
    y_metrics=[f"{m}_anom" for m in ["rzmc_inc_rms_mjj", "soil_water_abs_activity_mjj"] if f"{m}_anom" in analysisB_anom_table.columns],
    by_col="period",
    title="Analysis B: within-tile snow-activity anomaly vs SM-activity anomaly",
    xlabel="MAM snow_abs_netpack anomaly (kg m-2)",
    ylabel="Native water-increment activity anomaly (kg m-2 season-1)",
    secondary_y_metrics=["rzmc_inc_rms_mjj_anom"],
    secondary_ylabel="Diagnostic RZMC activity anomaly (m3 m-3)",
    stem="analysisB_binned_within_tile_anomaly_controls",
)
plot_binned_lines(
    analysisB_binned_signed,
    x_metric="snow_net_mam",
    y_metrics=[m for m in ["rzmc_inc_mean_mjj", "sfmc_inc_mean_mjj", "soil_water_net_approx_mjj"] if m in B_signed_targets],
    by_col="period",
    title="Analysis B: signed snow correction vs signed SM/soil correction",
    xlabel="MAM snow_net (kg m-2)",
    ylabel="Approx. soil-water correction (kg m-2 season-1)",
    secondary_y_metrics=["rzmc_inc_mean_mjj", "sfmc_inc_mean_mjj"],
    secondary_ylabel="Diagnostic SM correction (m3 m-3)",
    stem="analysisB_binned_signed_snow_vs_signed_sm_correction",
)


### Figure Notes: Analysis B Correlation Maps and Signed Hexbin

The next cell creates `analysisB_smapera_tilewise_correlation_maps` and `analysisB_smapera_signed_snow_vs_rzmc_hexbin`.

The correlation maps use SMAP-era clean years and the NH seasonal-snow mask. For each tile, Pearson correlation is computed across years, with at least 6 valid years required. The four pairs are MAM `snow_abs_netpack` versus MJJ `RZMC_INC_RMS`, MAM `snow_abs_netpack` versus MJJ `soil_water_abs_activity`, MAM `snow_net` versus MJJ `RZMC_INC_MEAN`, and MAM `snow_net` versus MJJ `soil_water_net_approx`. Each map value is correlation coefficient `r`, centered at zero. These maps are a cautious screen for repeated within-tile year-to-year coupling.

The hexbin plot uses SMAP-era NH seasonal-snow tile-years with finite `snow_net_mam` and `rzmc_inc_mean_mjj`. If more than 500,000 samples are available, it randomly samples 500,000 rows for plotting. The x-axis is signed MAM `snow_net` in `kg m-2`; the y-axis is MJJ `RZMC_ANA - RZMC_FCST` mean in `m3 m-3`. Hexbin color is log-count, and zero lines divide snow addition/removal from wetting/drying diagnostic correction quadrants.


In [ ]:
# Analysis B correlation maps and density/quadrant plot.
B_corr_rows = []
B_corr_maps = {}
for period_key, years in {"pre_smap_mw": PERIODS["pre_smap_mw"]["clean_years"], "smap_era": PERIODS["smap_era"]["clean_years"]}.items():
    arrays = B_ARRAYS[period_key]
    pairs = [
        ("snow_abs_netpack_mam", "rzmc_inc_rms_mjj"),
        ("snow_abs_netpack_mam", "soil_water_abs_activity_mjj"),
        ("snow_net_mam", "rzmc_inc_mean_mjj"),
        ("snow_net_mam", "soil_water_net_approx_mjj"),
    ]
    for x_name, y_name in pairs:
        if x_name not in arrays or y_name not in arrays:
            continue
        corr, count = corr_by_tile(arrays[x_name], arrays[y_name], min_years=6)
        corr = corr.where(seasonal_snow_mask).load()
        key = f"{period_key}_{x_name}_vs_{y_name}"
        B_corr_maps[key] = corr
        vals = corr.values[np.isfinite(corr.values)]
        B_corr_rows.append({
            "period": period_key,
            "x": x_name,
            "y": y_name,
            "n_tiles": int(vals.size),
            "median_corr": float(np.nanmedian(vals)) if vals.size else np.nan,
            "mean_corr": float(np.nanmean(vals)) if vals.size else np.nan,
            "q25_corr": float(np.nanpercentile(vals, 25)) if vals.size else np.nan,
            "q75_corr": float(np.nanpercentile(vals, 75)) if vals.size else np.nan,
        })
analysisB_corr_summary = pd.DataFrame(B_corr_rows)
analysisB_corr_summary.to_csv(OUT_DIR / "analysisB_tilewise_correlation_summary.csv", index=False)
display(analysisB_corr_summary)

smap_corr_items = []
for key, corr in B_corr_maps.items():
    if key.startswith("smap_era"):
        title = key.replace("smap_era_", "").replace("_", " ")
        smap_corr_items.append({"values": corr, "title": title, "metric": "correlation", "units": "r", "cmap": "RdBu_r", "norm": mcolors.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)})
if smap_corr_items:
    plot_map_grid(smap_corr_items[:4], seasonal_snow_mask, "Analysis B: SMAP-era tile-wise correlations across years", "analysisB_smapera_tilewise_correlation_maps", extent=[-180, 180, 20, 90], ncols=2)

quad_df = analysisB_table[(analysisB_table["period"] == "smap_era") & analysisB_table[["snow_net_mam", "rzmc_inc_mean_mjj"]].notnull().all(axis=1)].copy()
if not quad_df.empty:
    sample = quad_df.sample(n=min(len(quad_df), 500000), random_state=42)
    fig, ax = plt.subplots(figsize=(6.8, 5.2))
    hb = ax.hexbin(sample["snow_net_mam"], sample["rzmc_inc_mean_mjj"], gridsize=70, bins="log", mincnt=1, cmap="magma")
    ax.axhline(0, color="white", linewidth=0.9)
    ax.axvline(0, color="white", linewidth=0.9)
    cb = fig.colorbar(hb, ax=ax)
    cb.set_label("log10 count")
    ax.set_xlabel("MAM cumulative snow_net (kg m-2)")
    ax.set_ylabel("MJJ RZMC ANA-FCST mean (m3 m-3)")
    ax.set_title("Analysis B: SMAP-era signed snow correction vs signed RZMC correction")
    ax.grid(True, alpha=0.15)
    savefig(fig, "analysisB_smapera_signed_snow_vs_rzmc_hexbin")
    show_figure(fig)


## Analysis B-Control: Precip-Artifact Negative Control

Analysis B's post-2007 results sit in the window where `PRECTOTCORRLAND` differs between OL and DA at a level inconsistent with identical forcing (see the per-tile cumulative-drift investigation in the project discussion). This section bounds how much of Analysis B's signal that artifact alone could produce, by running the same pipeline with the precip difference standing in for the snow predictor.

- `dp_cumulative_mam` / `dp_abs_cumulative_mam`: seasonal `DA-OL PRECTOTCORRLAND` over MAM, built exactly like `snow_net_mam`/`snow_abs_netpack_mam`. Precipitation forcing should be common to OL and DA, so any swing this produces against B's response targets is attributable to the artifact, not genuine snow-DA-to-SM-DA coupling.
- Collinearity check: correlation between the snow predictors and the precip-artifact predictors at the tile-year level. Low correlation rules out the artifact directly biasing the snow predictor itself.
- Negative-control swings: the same lowest-minus-highest-bin swing computed for `dp_cumulative_mam`/`dp_abs_cumulative_mam` against B's exact response targets, compared directly to B's real snow-driven swings.
- Tercile stratification: B's signed `snow_net` relationship re-run within terciles of `dp_abs_cumulative_mam`. If the relationship holds inside every tercile, it is not being generated by the tiles/years with the largest precip-artifact noise.


In [ ]:
# Negative control: bound how much of Analysis B's signal the unsynced post-2007
# precipitation forcing could produce on its own. dP has no physical reason to
# drive soil-moisture DA correction, so any swing it produces through this
# identical pipeline is an empirical upper bound on the artifact's contribution.
B0_tables = []
for period_key in ["pre_smap_mw", "smap_era"]:
    years = PERIODS[period_key]["clean_years"]
    dp_mam = seasonal_delta_var("PRECTOTCORRLAND", years, "MAM").load()
    arrays = {
        "dp_cumulative_mam": dp_mam.rename("dp_cumulative_mam"),
        "dp_abs_cumulative_mam": abs(dp_mam).rename("dp_abs_cumulative_mam"),
    }
    B0_tables.append(table_from_year_arrays(years, arrays, seasonal_snow_mask, period=period_key))
analysisB0_table = pd.concat(B0_tables, ignore_index=True)

analysisB_with_control = analysisB_table.merge(
    analysisB0_table[["period", "tile", "year", "dp_cumulative_mam", "dp_abs_cumulative_mam"]],
    on=["period", "tile", "year"], how="left",
)

collinearity_rows = []
for period_key, g in analysisB_with_control.groupby("period"):
    sub = g[["snow_net_mam", "dp_cumulative_mam", "snow_abs_netpack_mam", "dp_abs_cumulative_mam"]].dropna()
    collinearity_rows.append({
        "period": period_key,
        "n": len(sub),
        "corr_signed_snow_net_vs_dP": sub["snow_net_mam"].corr(sub["dp_cumulative_mam"]),
        "corr_snow_activity_vs_dP_activity": sub["snow_abs_netpack_mam"].corr(sub["dp_abs_cumulative_mam"]),
    })
analysisB0_collinearity = pd.DataFrame(collinearity_rows)
analysisB0_collinearity.to_csv(OUT_DIR / "analysisB0_collinearity_check.csv", index=False)
display(analysisB0_collinearity)

B0_activity_bins = []
for target in B_activity_targets:
    out = binned_summary(analysisB_with_control, "dp_abs_cumulative_mam", target, by_cols=["period"], n_bins=N_BINS, bin_kind="precip_artifact_activity")
    if not out.empty:
        B0_activity_bins.append(out)
analysisB0_binned_activity = pd.concat(B0_activity_bins, ignore_index=True) if B0_activity_bins else pd.DataFrame()
analysisB0_binned_activity.to_csv(OUT_DIR / "analysisB0_binned_precip_artifact_activity.csv", index=False)

B0_signed_bins = []
for target in B_signed_targets:
    out = binned_summary(analysisB_with_control, "dp_cumulative_mam", target, by_cols=["period"], n_bins=5, bin_kind="precip_artifact_signed")
    if not out.empty:
        B0_signed_bins.append(out)
analysisB0_binned_signed = pd.concat(B0_signed_bins, ignore_index=True) if B0_signed_bins else pd.DataFrame()
analysisB0_binned_signed.to_csv(OUT_DIR / "analysisB0_binned_precip_artifact_signed.csv", index=False)

comparison_rows = []
for period_key in ["pre_smap_mw", "smap_era"]:
    for target in B_activity_targets:
        real_swing = high_low_signal(analysisB_binned_activity, "snow_abs_netpack_mam", target, period=period_key)
        control_swing = high_low_signal(analysisB0_binned_activity, "dp_abs_cumulative_mam", target, period=period_key)
        comparison_rows.append({
            "period": period_key, "target": target, "kind": "activity_magnitude",
            "snow_driven_swing": real_swing, "precip_artifact_swing": control_swing,
            "artifact_share_of_signal": (abs(control_swing) / abs(real_swing)) if real_swing else np.nan,
        })
    for target in B_signed_targets:
        real_swing = high_low_signal(analysisB_binned_signed, "snow_net_mam", target, period=period_key)
        control_swing = high_low_signal(analysisB0_binned_signed, "dp_cumulative_mam", target, period=period_key)
        comparison_rows.append({
            "period": period_key, "target": target, "kind": "signed",
            "snow_driven_swing": real_swing, "precip_artifact_swing": control_swing,
            "artifact_share_of_signal": (abs(control_swing) / abs(real_swing)) if real_swing else np.nan,
        })
analysisB0_comparison = pd.DataFrame(comparison_rows)
analysisB0_comparison.to_csv(OUT_DIR / "analysisB0_precip_artifact_bound_comparison.csv", index=False)
display(analysisB0_comparison)

max_share = analysisB0_comparison["artifact_share_of_signal"].replace([np.inf, -np.inf], np.nan).max()
if pd.notna(max_share):
    print(f"Largest precip-artifact share of any Analysis B signal (negative-control swing / real swing): {max_share:.1%}")
else:
    print("Could not compute artifact share (missing data).")


In [ ]:
def add_group_tercile(df, col, group_cols=("period",), label=None):
    label = label or f"{col}_tercile"
    out = df.copy()
    out[label] = np.nan
    for _, g in out.groupby(list(group_cols)):
        valid = g[col].dropna()
        if valid.size < 30:
            continue
        edges = np.unique(valid.quantile([0.0, 1 / 3, 2 / 3, 1.0]).values).astype(float)
        if edges.size < 4:
            continue
        edges[0] -= 1e-9
        edges[-1] += 1e-9
        out.loc[g.index, label] = pd.cut(g[col], bins=edges, labels=["low", "mid", "high"], include_lowest=True)
    return out

analysisB_with_control = add_group_tercile(analysisB_with_control, "dp_abs_cumulative_mam", label="dp_tercile")

B_stratified_bins = []
for tercile in ["low", "mid", "high"]:
    sub = analysisB_with_control[analysisB_with_control["dp_tercile"] == tercile]
    for target in B_signed_targets:
        out = binned_summary(sub, "snow_net_mam", target, by_cols=["period"], n_bins=5, bin_kind="signed_snow_net_within_dP_tercile")
        if not out.empty:
            out["dp_tercile"] = tercile
            B_stratified_bins.append(out)
analysisB_stratified_signed = pd.concat(B_stratified_bins, ignore_index=True) if B_stratified_bins else pd.DataFrame()
analysisB_stratified_signed.to_csv(OUT_DIR / "analysisB_signed_snow_relationship_within_dP_terciles.csv", index=False)

stratified_swing_rows = []
for period_key in ["pre_smap_mw", "smap_era"]:
    for target in B_signed_targets:
        for tercile in ["low", "mid", "high"]:
            sub = analysisB_stratified_signed[
                (analysisB_stratified_signed["period"] == period_key)
                & (analysisB_stratified_signed["dp_tercile"] == tercile)
                & (analysisB_stratified_signed["y_metric"] == target)
            ].sort_values("x_mean")
            if sub.empty:
                continue
            stratified_swing_rows.append({
                "period": period_key,
                "target": target,
                "dp_tercile": tercile,
                "swing": float(sub.iloc[-1]["y_mean"] - sub.iloc[0]["y_mean"]),
                "n_bins": len(sub),
            })
analysisB_stratified_swing_summary = pd.DataFrame(stratified_swing_rows)
analysisB_stratified_swing_summary.to_csv(OUT_DIR / "analysisB_signed_swing_by_dP_tercile.csv", index=False)
display(analysisB_stratified_swing_summary)

plot_targets = [t for t in ["rzmc_inc_mean_mjj", "soil_water_net_approx_mjj"] if t in B_signed_targets]
if plot_targets:
    fig, axes = plt.subplots(len(plot_targets), 2, figsize=(11.5, 4.4 * len(plot_targets)), squeeze=False)
    for row, target in enumerate(plot_targets):
        for col, period_key in enumerate(["pre_smap_mw", "smap_era"]):
            ax = axes[row][col]
            sub = analysisB_stratified_signed[(analysisB_stratified_signed["period"] == period_key) & (analysisB_stratified_signed["y_metric"] == target)]
            for tercile, sg in sub.groupby("dp_tercile"):
                sg = sg.sort_values("x_mean")
                ax.errorbar(sg["x_mean"], sg["y_mean"], yerr=sg["y_se"], marker="o", linewidth=1.3, capsize=2, label=f"dP-artifact tercile: {tercile}")
            ax.axhline(0, color="0.2", linewidth=0.8)
            ax.set_xlabel(metric_label("snow_net_mam", include_units=True, context="seasonal"))
            ax.set_ylabel(metric_label(target, include_units=True, context="seasonal"))
            ax.set_title(PERIODS[period_key]["label"], fontsize=9)
            ax.grid(True, alpha=0.25)
            ax.legend(fontsize=7)
    fig.suptitle("Analysis B-control: does the signed snow-to-SM relationship survive controlling for the precip artifact?")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    savefig(fig, "analysisB_control_signed_relationship_within_dP_terciles")
    show_figure(fig)


## Analysis C: Hydrology and Energy Partitioning

This analysis explores model-internal water and energy responses. It is not independent validation. The warm-season pathway uses JJA SMAP-era warm snow-free grid-cell-years; the snow-energy pathway uses MAM snow increments and AMJ/MJJ transition-season energy responses.


### Figure Notes: Analysis C1/C2 Hydro-Energy Plots

The next cell creates the Analysis C1 warm-season hydro-energy figures and the Analysis C2 snow-to-energy binned figure.

`analysisC1_warmseason_energy_partitioning_maps` uses SMAP-era clean years and a dynamic warm snow-free JJA mask. A tile-year is included only where both OL and DA have JJA `FRLANDSNO < 0.05`, both have `TSOIL1 > 277.15 K`, and the tile is valid land. Each mapped field is the clean-year mean of JJA DA-OL response. Panels include `RZMC` and `SFMC` (`m3 m-3`), `EVLAND` (`kg m-2 season-1`), `LHLAND`, `SHLAND`, `LHLANDTRNS`, `LHLANDSOIL`, `LHLANDINTR` (`W m-2`), and `EF` (fraction). `EF` is computed separately as `LHLAND / (LHLAND + SHLAND)` for DA and OL, then differenced.

`analysisC1_binned_rzmc_vs_energy_partitioning` uses the same SMAP-era warm snow-free JJA tile-year table. The x-axis is JJA `dRZMC` in `m3 m-3`, split into 8 quantile bins. The primary y-axis contains energy/ET responses (`dLHLAND`, `dSHLAND`, `dEVLAND`, `dLHLANDTRNS`, `dLHLANDSOIL`), with heat fluxes in `W m-2` and ET as a seasonal water total. The secondary y-axis contains `dEF` as a fraction. This figure tests whether DA-induced root-zone wetting corresponds to higher latent heat/ET and lower sensible heat.

`analysisC2_binned_snow_activity_vs_energy_response` uses NH seasonal-snow tile-years in the MODIS-only and SMAP-era periods. The x-axis is MAM `snow_abs_netpack` in `kg m-2`, split into 8 quantile bins within each period. The primary y-axis contains MJJ energy responses such as `dSWLAND`, `dGHLAND`, and `dLHLANDSBLN` in `W m-2`. The secondary y-axis contains incompatible-scale variables, `dTSOIL1` in `K` and `dEF` as a fraction. This is exploratory: it tests whether stronger MAM snow correction is associated with transition-season energy changes.


In [ ]:
# C1: warm-season soil-moisture pathway.
years_C1 = PERIODS["smap_era"]["clean_years"]
warm_jja_mask = warm_snowfree_season_mask(years_C1, "JJA")
C1_arrays = {}
for var in ["RZMC", "SFMC", "EVLAND", "LHLAND", "SHLAND", "EF", "LHLANDTRNS", "LHLANDSOIL", "LHLANDINTR", "TSOIL1"]:
    try:
        C1_arrays[f"d{var.lower()}_jja"] = seasonal_response(var, years_C1, "JJA").load()
    except Exception as exc:
        warnings.warn(f"Analysis C1 skipped {var}: {exc}")
C1_table = table_from_year_arrays(years_C1, C1_arrays, warm_jja_mask, period="smap_era")
C1_summary = pd.DataFrame([{"analysis": "C1", "period": "smap_era", "n_rows": len(C1_table), "domain": "dynamic warm snow-free JJA"}])
C1_summary.to_csv(OUT_DIR / "analysisC1_sample_summary.csv", index=False)
display(C1_summary)
display(C1_table.head())

C1_targets = [c for c in ["def_jja", "dlhland_jja", "dshland_jja", "devland_jja", "dlhlandtrns_jja", "dlhlandsoil_jja"] if c in C1_table.columns]
C1_bins = []
for target in C1_targets:
    out = binned_summary(C1_table, "drzmc_jja", target, n_bins=N_BINS, bin_kind="rzmc_response")
    if not out.empty:
        C1_bins.append(out)
analysisC1_binned = pd.concat(C1_bins, ignore_index=True) if C1_bins else pd.DataFrame()
analysisC1_binned.to_csv(OUT_DIR / "analysisC1_binned_rzmc_vs_energy_partitioning.csv", index=False)
display(analysisC1_binned.head(30))

C1_map_items = []
for var, title in [
    ("drzmc_jja", "JJA DA-OL RZMC"),
    ("dsfmc_jja", "JJA DA-OL SFMC"),
    ("devland_jja", "JJA DA-OL ET"),
    ("dlhland_jja", "JJA DA-OL latent heat"),
    ("dshland_jja", "JJA DA-OL sensible heat"),
    ("def_jja", "JJA DA-OL EF"),
    ("dlhlandtrns_jja", "JJA DA-OL transpiration LH"),
    ("dlhlandsoil_jja", "JJA DA-OL bare-soil evap LH"),
    ("dlhlandintr_jja", "JJA DA-OL interception LH"),
]:
    if var in C1_arrays:
        C1_map_items.append({"values": C1_arrays[var].where(warm_jja_mask).mean("year", skipna=True).load(), "title": title, "metric": var, "cmap": "RdBu_r"})
if C1_map_items:
    plot_map_grid(C1_map_items, valid_land_mask, "Analysis C1: SMAP-era warm-season hydro-energy response", "analysisC1_warmseason_energy_partitioning_maps", extent=[-180, 180, -60, 80], ncols=3)

plot_binned_lines(
    analysisC1_binned,
    x_metric="drzmc_jja",
    y_metrics=[m for m in ["def_jja", "dlhland_jja", "dshland_jja", "devland_jja", "dlhlandtrns_jja", "dlhlandsoil_jja"] if m in C1_targets],
    title="Analysis C1: JJA DA-OL RZMC vs energy/ET response, SMAP era",
    xlabel="JJA DA-OL RZMC (m3 m-3)",
    ylabel="Primary response (W m-2 or kg m-2 season-1)",
    secondary_y_metrics=["def_jja"],
    secondary_ylabel="Evaporative fraction response (fraction)",
    stem="analysisC1_binned_rzmc_vs_energy_partitioning",
)

# C2: snow pathway into energy partitioning.
C2_rows = []
C2_bins = []
for period_key in ["modis_only", "smap_era"]:
    years = PERIODS[period_key]["clean_years"]
    arrays = {
        "snow_net_mam": seasonal_catch_var("snow_net", years, "MAM").load(),
        "snow_abs_netpack_mam": seasonal_catch_var("snow_abs_netpack", years, "MAM").load(),
        "dFRLANDSNO_mam": seasonal_response("FRLANDSNO", years, "MAM").load(),
        "dSNOMASLAND_mam": seasonal_response("SNOMASLAND", years, "MAM").load(),
    }
    for season in ["AMJ", "MJJ"]:
        for var in ["SWLAND", "LWLAND", "GHLAND", "TSOIL1", "LHLANDSBLN", "LHLAND", "SHLAND", "EF"]:
            try:
                arrays[f"d{var.lower()}_{season.lower()}"] = seasonal_response(var, years, season).load()
            except Exception as exc:
                warnings.warn(f"Analysis C2 skipped {period_key} {var} {season}: {exc}")
    table = table_from_year_arrays(years, arrays, seasonal_snow_mask, period=period_key)
    C2_rows.append({"period": period_key, "n_rows": len(table), "n_years": len(years)})
    for target in [c for c in ["dswland_mjj", "dghland_mjj", "dtsoil1_mjj", "dlhlandsbln_mjj", "def_mjj"] if c in table.columns]:
        out = binned_summary(table, "snow_abs_netpack_mam", target, by_cols=["period"], n_bins=N_BINS, bin_kind="snow_activity_energy")
        if not out.empty:
            C2_bins.append(out)
analysisC2_sample_summary = pd.DataFrame(C2_rows)
analysisC2_sample_summary.to_csv(OUT_DIR / "analysisC2_sample_summary.csv", index=False)
analysisC2_binned = pd.concat(C2_bins, ignore_index=True) if C2_bins else pd.DataFrame()
analysisC2_binned.to_csv(OUT_DIR / "analysisC2_binned_snow_activity_vs_energy_response.csv", index=False)
display(analysisC2_sample_summary)
display(analysisC2_binned.head(30))

plot_binned_lines(
    analysisC2_binned,
    x_metric="snow_abs_netpack_mam",
    y_metrics=[m for m in ["dswland_mjj", "dghland_mjj", "dtsoil1_mjj", "dlhlandsbln_mjj", "def_mjj"] if m in analysisC2_binned.get("y_metric", pd.Series(dtype=str)).unique()],
    by_col="period",
    title="Analysis C2: MAM snow DA activity vs transition-season energy response",
    xlabel="MAM snow_abs_netpack (kg m-2)",
    ylabel="Energy response (W m-2)",
    secondary_y_metrics=["dtsoil1_mjj", "def_mjj"],
    secondary_ylabel="Temperature / EF response (K or fraction)",
    stem="analysisC2_binned_snow_activity_vs_energy_response",
)


### Figure Notes: Analysis C3 Latent-Heat Component Climatologies

The next cell creates `analysisC3_latent_component_delta_climatologies`. It starts from monthly area-weighted domain means, then averages by calendar month within each observing-system period.

The plotted values are monthly climatological `DA - OL` latent heat components in `W m-2`: `LHLANDTRNS` (transpiration), `LHLANDSOIL` (bare-soil evaporation), `LHLANDINTR` (interception evaporation), `LHLANDSBLN` (snow sublimation), and total `LHLAND`. The six panels are NH seasonal snow and warm static snow-free domains for the MODIS-only, microwave pre-SMAP, and SMAP-era periods. The x-axis is calendar month. This figure separates whether the total latent-heat response is coming from transpiration, soil evaporation, interception, or snow sublimation.


In [ ]:
# C3: latent heat component decomposition as monthly climatologies.
latent_components = ["LHLANDTRNS", "LHLANDSOIL", "LHLANDINTR", "LHLANDSBLN", "LHLAND"]
domain_masks = {
    "global_land": valid_land_mask,
    "nh_seasonal_snow": seasonal_snow_mask,
    "high_snow_da_activity": high_snow_activity_mask,
    "warm_static_snowfree": warm_static_mask,
}

C3_rows = []
for period_key, period in PERIODS.items():
    for domain_name, domain_mask in domain_masks.items():
        for var in latent_components:
            if var not in VAR_SOURCE:
                continue
            for exp in ["ol", "da"]:
                series = weighted_tile_mean(monthly_model_var(exp, var), domain_mask).load()
                series = series.sel(time=slice(period["start"], period["end"]))
                for month in range(1, 13):
                    vals = series.sel(time=series.time.dt.month == month).values
                    C3_rows.append({
                        "period": period_key,
                        "domain": domain_name,
                        "experiment": exp.upper(),
                        "variable": var,
                        "month": month,
                        "mean": float(np.nanmean(vals)) if np.isfinite(vals).any() else np.nan,
                        "n_months": int(np.isfinite(vals).sum()),
                        "weighted": use_area_weights,
                    })
analysisC3_climatology = pd.DataFrame(C3_rows)
analysisC3_delta = analysisC3_climatology.pivot_table(index=["period", "domain", "variable", "month"], columns="experiment", values="mean").reset_index()
if {"DA", "OL"}.issubset(analysisC3_delta.columns):
    analysisC3_delta["DA_minus_OL"] = analysisC3_delta["DA"] - analysisC3_delta["OL"]
analysisC3_climatology.to_csv(OUT_DIR / "analysisC3_latent_component_monthly_climatology_ol_da.csv", index=False)
analysisC3_delta.to_csv(OUT_DIR / "analysisC3_latent_component_monthly_climatology_delta.csv", index=False)
display(analysisC3_delta.head(30))

plot_delta = analysisC3_delta[(analysisC3_delta["domain"].isin(["nh_seasonal_snow", "warm_static_snowfree"])) & (analysisC3_delta["variable"].isin(latent_components))]
if not plot_delta.empty and "DA_minus_OL" in plot_delta:
    fig, axes = plt.subplots(2, 3, figsize=(15, 7.2), sharex=True)
    axes = axes.ravel()
    combos = [("nh_seasonal_snow", "modis_only"), ("nh_seasonal_snow", "pre_smap_mw"), ("nh_seasonal_snow", "smap_era"), ("warm_static_snowfree", "modis_only"), ("warm_static_snowfree", "pre_smap_mw"), ("warm_static_snowfree", "smap_era")]
    for ax, (domain_name, period_key) in zip(axes, combos):
        sub = plot_delta[(plot_delta["domain"] == domain_name) & (plot_delta["period"] == period_key)]
        for var, sv in sub.groupby("variable"):
            ax.plot(sv["month"], sv["DA_minus_OL"], marker="o", linewidth=1.2, label=var)
        ax.axhline(0, color="0.2", linewidth=0.7)
        ax.set_title(f"{domain_name}, {period_key}", fontsize=9)
        ax.grid(True, alpha=0.25)
    axes[0].set_ylabel("DA-OL latent heat (W m-2)")
    axes[3].set_ylabel("DA-OL latent heat (W m-2)")
    for ax in axes[3:]:
        ax.set_xlabel("month")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=5, fontsize=8)
    fig.suptitle("Analysis C3: latent heat component DA-OL monthly climatologies")
    fig.subplots_adjust(bottom=0.16, top=0.88)
    savefig(fig, "analysisC3_latent_component_delta_climatologies")
    show_figure(fig)


## Analysis D: Observing-System Evolution of DA Activity and Propagated Response

Monthly domain-mean time series show whether DA activity and propagated hydrologic/energy responses evolve coherently with observing-system transitions. Domain means are area-weighted if tile areas are available.


### Figure Notes: Analysis D Monthly Time Series

The next cell creates `analysisD_monthly_timeseries_nh_seasonal_snow`. It shows monthly area-weighted domain means for the NH seasonal-snow mask. Background shading marks the MODIS-only, microwave pre-SMAP, and SMAP-era periods.

The top panel, `DA activity`, plots monthly `snow_abs_netpack` and `soil_water_abs_activity` on the primary axis in `kg m-2 month-1`. The secondary axis plots `RZMC_INC_RMS` in `m3 m-3`, a diagnostic volumetric ANA-FCST activity metric.

The middle panel, `Hydrologic response`, plots `dTWLAND`, `dEVLAND`, and `dTOTAL_RUNOFF` on the primary axis. `dEVLAND` and `dTOTAL_RUNOFF` are monthly flux totals in `kg m-2 month-1`; `dTWLAND` is total water storage in `kg m-2`. The secondary axis plots monthly `dRZMC` in `m3 m-3`.

The bottom panel, `Energy response`, plots monthly `dLHLAND` and `dSHLAND` in `W m-2` on the primary axis and `dEF` as a fraction on the secondary axis. This figure is for observing-system evolution and seasonal phasing, not tile-year binning.


In [ ]:
D_metrics = {
    "increment_activity": {
        "snow_abs_netpack": CATCH["snow_abs_netpack"],
        "snow_net": CATCH["snow_net"],
        "soil_water_abs_activity": CATCH["soil_water_abs_activity"],
        "RZMC_INC_RMS": INST3["RZMC_INC_RMS"],
        "RZMC_INC_ABS_MEAN": INST3["RZMC_INC_ABS_MEAN"],
        "SFMC_INC_RMS": INST3["SFMC_INC_RMS"],
        "SFMC_INC_ABS_MEAN": INST3["SFMC_INC_ABS_MEAN"],
    },
    "hydrology_response": {
        "dSFMC": monthly_response("SFMC"),
        "dRZMC": monthly_response("RZMC"),
        "dTWLAND": monthly_response("TWLAND"),
        "dEVLAND": monthly_response("EVLAND"),
        "dTOTAL_RUNOFF": monthly_response("TOTAL_RUNOFF"),
        "dBASEFLOWLAND": monthly_response("BASEFLOWLAND"),
        "dQINFILLAND": monthly_response("QINFILLAND"),
        "dSMLAND": monthly_response("SMLAND"),
    },
    "energy_response": {
        "dLHLAND": monthly_response("LHLAND"),
        "dSHLAND": monthly_response("SHLAND"),
        "dEF": monthly_response("EF"),
        "dGHLAND": monthly_response("GHLAND"),
        "dTSOIL1": monthly_response("TSOIL1"),
    },
}

D_rows = []
for domain_name, domain_mask in domain_masks.items():
    for family, metrics in D_metrics.items():
        for metric_name, da in metrics.items():
            series = weighted_tile_mean(da, domain_mask).load()
            for t, value in zip(pd.to_datetime(series.time.values), series.values):
                D_rows.append({
                    "time": t,
                    "domain": domain_name,
                    "family": family,
                    "metric": metric_name,
                    "value": float(value) if np.isfinite(value) else np.nan,
                    "weighted": use_area_weights,
                })
analysisD_timeseries = pd.DataFrame(D_rows)
analysisD_timeseries.to_csv(OUT_DIR / "analysisD_domain_monthly_timeseries.csv", index=False)
display(analysisD_timeseries.head(30))

analysisD_period_means = []
for period_key, period in PERIODS.items():
    t0 = pd.Timestamp(period["start"])
    t1 = pd.Timestamp(period["end"])
    sub = analysisD_timeseries[(analysisD_timeseries["time"] >= t0) & (analysisD_timeseries["time"] <= t1)]
    period_summary = sub.groupby(["domain", "family", "metric"], dropna=False)["value"].agg(["mean", "median", "std", "count"]).reset_index()
    period_summary["period"] = period_key
    analysisD_period_means.append(period_summary)
analysisD_period_summary = pd.concat(analysisD_period_means, ignore_index=True)
analysisD_period_summary.to_csv(OUT_DIR / "analysisD_period_mean_summary.csv", index=False)
display(analysisD_period_summary.head(30))

plot_domain = "nh_seasonal_snow"
fig, axes = plt.subplots(3, 1, figsize=(13, 9.0), sharex=True)
plot_sets = [
    {
        "family": "increment_activity",
        "primary": ["snow_abs_netpack", "soil_water_abs_activity"],
        "secondary": ["RZMC_INC_RMS"],
        "title": "DA activity",
        "ylabel": "Increment activity\n(kg m-2 month-1)",
        "secondary_ylabel": "RZMC RMS\n(m3 m-3)",
    },
    {
        "family": "hydrology_response",
        "primary": ["dTWLAND", "dEVLAND", "dTOTAL_RUNOFF"],
        "secondary": ["dRZMC"],
        "title": "Hydrologic response",
        "ylabel": "Water response\n(kg m-2; monthly flux totals)",
        "secondary_ylabel": "RZMC response\n(m3 m-3)",
    },
    {
        "family": "energy_response",
        "primary": ["dLHLAND", "dSHLAND"],
        "secondary": ["dEF"],
        "title": "Energy response",
        "ylabel": "Turbulent heat flux\n(W m-2)",
        "secondary_ylabel": "EF response\n(fraction)",
    },
]
for ax, spec in zip(axes, plot_sets):
    family = spec["family"]
    sub = analysisD_timeseries[(analysisD_timeseries["domain"] == plot_domain) & (analysisD_timeseries["family"] == family)]
    for metric in spec["primary"]:
        sm = sub[sub["metric"] == metric]
        if sm.empty:
            continue
        ax.plot(sm["time"], sm["value"], linewidth=1.2, label=metric_label(metric, include_units=True, context="monthly"))
    ax2 = ax.twinx() if spec["secondary"] else None
    if ax2 is not None:
        for metric in spec["secondary"]:
            sm = sub[sub["metric"] == metric]
            if sm.empty:
                continue
            ax2.plot(sm["time"], sm["value"], linewidth=1.1, linestyle="--", label=metric_label(metric, include_units=True, context="monthly"), color="0.2")
        ax2.axhline(0, color="0.45", linewidth=0.6, linestyle=":")
        ax2.set_ylabel(spec["secondary_ylabel"])
    ax.axhline(0, color="0.2", linewidth=0.7)
    for period_key, period in PERIODS.items():
        ax.axvspan(pd.Timestamp(period["start"]), pd.Timestamp(period["end"]), color=period["color"], alpha=0.08)
    ax.set_title(spec["title"])
    ax.set_ylabel(spec["ylabel"])
    ax.grid(True, alpha=0.25)
    handles, labels = ax.get_legend_handles_labels()
    if ax2 is not None:
        h2, l2 = ax2.get_legend_handles_labels()
        handles += h2
        labels += l2
    ax.legend(handles, labels, loc="upper left", fontsize=8, ncol=2)
axes[-1].set_xlabel("time")
fig.suptitle("Analysis D: monthly DA activity and propagated responses, NH seasonal-snow domain")
fig.tight_layout(rect=[0.035, 0, 0.965, 0.95])
fig.subplots_adjust(left=0.11, right=0.88, hspace=0.36, top=0.92)
savefig(fig, "analysisD_monthly_timeseries_nh_seasonal_snow")
show_figure(fig)


### Figure Notes: Percent-Change Companions for Analysis A/C/D

The next cell creates percent-change companion figures for Analysis A, C, and D. These figures use `100 * (DA - OL) / OL`, with small-denominator masking so near-zero OL values do not dominate the maps or binned summaries.

`analysisA_process_chain_maps_percent` keeps the MAM snow-DA predictor panels in native increment units because they have no OL baseline, then maps the AMJ/MJJ hydrologic responses as percent changes relative to OL.

`analysisA_binned_snow_activity_to_response_magnitude_percent` uses the same MODIS-only seasonal-snow tile-year sample as the absolute Analysis A binned figure. The x-axis remains MAM `snow_abs_netpack`; the y-axes are absolute percent-change response magnitudes relative to OL.

`analysisC1_warmseason_energy_partitioning_maps_percent` uses the same SMAP-era clean years and dynamic warm snow-free JJA mask as the absolute Analysis C maps. Each panel is the clean-year mean of tile-year percent change relative to OL. `EF` is still computed separately for DA and OL, then converted to a relative percent change using OL EF as the denominator.

`analysisC1_binned_rzmc_vs_energy_partitioning_percent` uses the same tile-year sample as the absolute binned Analysis C figure, but the x-axis is percent change in JJA RZMC and the y-axes are percent changes in EF, latent heat, sensible heat, ET, and latent-heat components.

`analysisD_monthly_timeseries_nh_seasonal_snow_percent` keeps the top DA-activity panel in native units because those increment/activity metrics do not have an OL baseline. The hydrology and energy panels are percent changes relative to OL.


In [ ]:
# Percent-change companion figures for Analysis A, C, and D.
# Formula: 100 * (DA - OL) / OL, with small-denominator masking.
PCT_DENOMINATOR_THRESHOLDS = {
    "SFMC": 0.01,
    "RZMC": 0.01,
    "PRMC": 0.01,
    "FRLANDSNO": 0.01,
    "SNOMASLAND": 1.0,
    "SNODPLAND": 0.01,
    "TWLAND": 1.0,
    "EVLAND": 1.0,
    "TOTAL_RUNOFF": 1.0,
    "RUNSURFLAND": 1.0,
    "BASEFLOWLAND": 1.0,
    "QINFILLAND": 1.0,
    "SMLAND": 1.0,
    "WCHANGELAND": 1.0,
    "LHLAND": 1.0,
    "SHLAND": 1.0,
    "GHLAND": 1.0,
    "SWLAND": 1.0,
    "LWLAND": 1.0,
    "LHLANDTRNS": 1.0,
    "LHLANDSOIL": 1.0,
    "LHLANDINTR": 1.0,
    "LHLANDSBLN": 1.0,
    "EF": 0.01,
    "TSOIL1": 200.0,
}


def pct_threshold(name: str) -> float:
    return PCT_DENOMINATOR_THRESHOLDS.get(name.upper(), 1.0)


def percent_change(da, ol, name: str):
    out = 100.0 * (da - ol) / ol.where(np.abs(ol) >= pct_threshold(name))
    out.name = f"pct_{name}"
    out.attrs["units"] = "%"
    out.attrs["long_name"] = f"100 * (DA - OL) / OL for {name}"
    return out


def seasonal_percent_response(name: str, years, season: str):
    if name in DERIVED_VARS:
        da = seasonal_model_derived("da", name, years, season)
        ol = seasonal_model_derived("ol", name, years, season)
    else:
        da = seasonal_model_var("da", name, years, season)
        ol = seasonal_model_var("ol", name, years, season)
    out = percent_change(da, ol, name)
    out.name = f"pct_{name}_{season}"
    return out


def monthly_percent_response(name: str):
    if name in {"TOTAL_RUNOFF", "EF"}:
        da = monthly_model_derived("da", name)
        ol = monthly_model_derived("ol", name)
    else:
        da = monthly_model_var("da", name)
        ol = monthly_model_var("ol", name)
    out = percent_change(da, ol, name)
    out.name = f"pct_{name}"
    return out


def percent_metric_label(metric: str) -> str:
    name = str(metric)
    clean = name
    if clean.startswith("pct_"):
        clean = clean[4:]
    for season in [s.lower() for s in SEASON_WINDOWS]:
        if clean.endswith(f"_{season}"):
            clean = clean[: -(len(season) + 1)]
            break
    return f"{DISPLAY_LABELS.get(clean.lower(), clean)} (% change vs OL)"


# Analysis A percent-change maps and binned relationship.
# Snow increment/activity predictors remain in native units because they have no OL baseline.
A_pct_arrays = {}
for season, variables in [
    ("AMJ", ["SMLAND", "QINFILLAND", "FRLANDSNO", "SNOMASLAND"]),
    ("MJJ", ["RZMC", "SFMC", "TWLAND", "EVLAND", "TOTAL_RUNOFF", "LHLAND", "SHLAND", "EF"]),
]:
    for var in variables:
        try:
            A_pct_arrays[f"pct_{var.lower()}_{season.lower()}"] = seasonal_percent_response(var, years_A, season).load()
        except Exception as exc:
            warnings.warn(f"Analysis A percent skipped {var} {season}: {exc}")

A_pct_table_arrays = {
    "snow_net_mam": A_arrays["snow_net_mam"],
    "snow_abs_netpack_mam": A_arrays["snow_abs_netpack_mam"],
    **A_pct_arrays,
}
A_pct_table = table_from_year_arrays(years_A, A_pct_table_arrays, seasonal_snow_mask, period="modis_only")
A_pct_summary = pd.DataFrame([{
    "analysis": "A_percent",
    "period": "modis_only",
    "n_rows": len(A_pct_table),
    "n_years": len(years_A),
    "n_tiles_static_mask": int(seasonal_snow_mask.sum()),
    "domain": "NH seasonal snow",
    "formula": "100 * (DA - OL) / OL for model response variables",
}])
A_pct_summary.to_csv(OUT_DIR / "analysisA_percent_sample_summary.csv", index=False)
display(A_pct_summary)
display(A_pct_table.head())

A_pct_targets = [
    c for c in [
        "pct_smland_amj", "pct_qinfilland_amj", "pct_rzmc_mjj", "pct_evland_mjj",
        "pct_total_runoff_mjj", "pct_twland_mjj", "pct_lhland_mjj", "pct_shland_mjj", "pct_ef_mjj",
    ]
    if c in A_pct_table.columns
]
A_pct_activity_targets = []
for c in A_pct_targets:
    A_pct_table[f"abs_{c}"] = np.abs(A_pct_table[c])
    A_pct_activity_targets.append(f"abs_{c}")

A_pct_activity_bins = []
for target in A_pct_activity_targets:
    out = binned_summary(A_pct_table, "snow_abs_netpack_mam", target, n_bins=N_BINS, bin_kind="snow_activity_percent_response")
    if not out.empty:
        A_pct_activity_bins.append(out)
analysisA_binned_activity_percent = pd.concat(A_pct_activity_bins, ignore_index=True) if A_pct_activity_bins else pd.DataFrame()
analysisA_binned_activity_percent.to_csv(OUT_DIR / "analysisA_binned_snow_activity_to_response_magnitude_percent.csv", index=False)
display(analysisA_binned_activity_percent.head(30))

A_pct_map_items = [
    {"values": A_arrays["snow_net_mam"].mean("year"), "title": "MAM snow_net", "metric": "snow_net_mam", "cmap": "RdBu_r"},
    {"values": A_arrays["snow_abs_netpack_mam"].mean("year"), "title": "MAM snow_abs_netpack", "metric": "snow_abs_netpack_mam", "cmap": "magma", "positive": True},
]
for var, title in [
    ("pct_smland_amj", "AMJ snowmelt % change"),
    ("pct_qinfilland_amj", "AMJ infiltration % change"),
    ("pct_rzmc_mjj", "MJJ RZMC % change"),
    ("pct_evland_mjj", "MJJ ET % change"),
    ("pct_total_runoff_mjj", "MJJ total runoff % change"),
    ("pct_twland_mjj", "MJJ total water % change"),
]:
    if var in A_pct_arrays:
        A_pct_map_items.append({
            "values": A_pct_arrays[var].where(seasonal_snow_mask).mean("year", skipna=True).load(),
            "title": title,
            "metric": var,
            "units": "%",
            "cmap": "RdBu_r",
        })
if A_pct_map_items:
    plot_map_grid(
        A_pct_map_items,
        seasonal_snow_mask,
        "Analysis A percent change: MODIS-only snow DA carryover, 2001-2006",
        "analysisA_process_chain_maps_percent",
        extent=[-180, 180, 20, 90],
        ncols=4,
    )

plot_binned_lines(
    analysisA_binned_activity_percent,
    x_metric="snow_abs_netpack_mam",
    y_metrics=[m for m in ["abs_pct_rzmc_mjj", "abs_pct_evland_mjj", "abs_pct_total_runoff_mjj", "abs_pct_twland_mjj"] if m in A_pct_activity_targets],
    title="Analysis A percent change: MAM snow DA activity vs response magnitude",
    xlabel="MAM snow_abs_netpack (kg m-2)",
    ylabel="Absolute response (% change vs OL)",
    secondary_y_metrics=[],
    stem="analysisA_binned_snow_activity_to_response_magnitude_percent",
)


# Analysis C1 percent-change maps and binned relationship.
C1_pct_arrays = {}
for var in ["RZMC", "SFMC", "EVLAND", "LHLAND", "SHLAND", "EF", "LHLANDTRNS", "LHLANDSOIL", "LHLANDINTR"]:
    try:
        C1_pct_arrays[f"pct_{var.lower()}_jja"] = seasonal_percent_response(var, years_C1, "JJA").load()
    except Exception as exc:
        warnings.warn(f"Analysis C1 percent skipped {var}: {exc}")

C1_pct_table = table_from_year_arrays(years_C1, C1_pct_arrays, warm_jja_mask, period="smap_era")
C1_pct_summary = pd.DataFrame([{
    "analysis": "C1_percent",
    "period": "smap_era",
    "n_rows": len(C1_pct_table),
    "domain": "dynamic warm snow-free JJA",
    "formula": "100 * (DA - OL) / OL",
}])
C1_pct_summary.to_csv(OUT_DIR / "analysisC1_percent_sample_summary.csv", index=False)
display(C1_pct_summary)
display(C1_pct_table.head())

C1_pct_targets = [c for c in ["pct_ef_jja", "pct_lhland_jja", "pct_shland_jja", "pct_evland_jja", "pct_lhlandtrns_jja", "pct_lhlandsoil_jja"] if c in C1_pct_table.columns]
C1_pct_bins = []
for target in C1_pct_targets:
    out = binned_summary(C1_pct_table, "pct_rzmc_jja", target, n_bins=N_BINS, bin_kind="rzmc_percent_response")
    if not out.empty:
        C1_pct_bins.append(out)
analysisC1_binned_percent = pd.concat(C1_pct_bins, ignore_index=True) if C1_pct_bins else pd.DataFrame()
analysisC1_binned_percent.to_csv(OUT_DIR / "analysisC1_binned_rzmc_vs_energy_partitioning_percent.csv", index=False)
display(analysisC1_binned_percent.head(30))

C1_pct_map_items = []
for var, title in [
    ("pct_rzmc_jja", "JJA RZMC % change"),
    ("pct_sfmc_jja", "JJA SFMC % change"),
    ("pct_evland_jja", "JJA ET % change"),
    ("pct_lhland_jja", "JJA latent heat % change"),
    ("pct_shland_jja", "JJA sensible heat % change"),
    ("pct_ef_jja", "JJA EF % change"),
    ("pct_lhlandtrns_jja", "JJA transpiration LH % change"),
    ("pct_lhlandsoil_jja", "JJA bare-soil evap LH % change"),
    ("pct_lhlandintr_jja", "JJA interception LH % change"),
]:
    if var in C1_pct_arrays:
        C1_pct_map_items.append({
            "values": C1_pct_arrays[var].where(warm_jja_mask).mean("year", skipna=True).load(),
            "title": title,
            "metric": var,
            "units": "%",
            "cmap": "RdBu_r",
        })
if C1_pct_map_items:
    plot_map_grid(
        C1_pct_map_items,
        valid_land_mask,
        "Analysis C1 percent change: SMAP-era warm-season hydro-energy response",
        "analysisC1_warmseason_energy_partitioning_maps_percent",
        extent=[-180, 180, -60, 80],
        ncols=3,
    )

plot_binned_lines(
    analysisC1_binned_percent,
    x_metric="pct_rzmc_jja",
    y_metrics=[m for m in ["pct_ef_jja", "pct_lhland_jja", "pct_shland_jja", "pct_evland_jja", "pct_lhlandtrns_jja", "pct_lhlandsoil_jja"] if m in C1_pct_targets],
    title="Analysis C1 percent change: JJA RZMC vs energy/ET response, SMAP era",
    xlabel="JJA RZMC percent change vs OL (%)",
    ylabel="Energy/ET percent change vs OL (%)",
    secondary_y_metrics=[],
    stem="analysisC1_binned_rzmc_vs_energy_partitioning_percent",
)


# Analysis D percent-change monthly companion. Increment activity remains native because it has no OL baseline.
D_percent_metrics = {
    "hydrology_percent_response": {
        "pctTWLAND": monthly_percent_response("TWLAND"),
        "pctEVLAND": monthly_percent_response("EVLAND"),
        "pctTOTAL_RUNOFF": monthly_percent_response("TOTAL_RUNOFF"),
        "pctRZMC": monthly_percent_response("RZMC"),
    },
    "energy_percent_response": {
        "pctLHLAND": monthly_percent_response("LHLAND"),
        "pctSHLAND": monthly_percent_response("SHLAND"),
        "pctEF": monthly_percent_response("EF"),
    },
}

D_pct_rows = []
for domain_name, domain_mask in domain_masks.items():
    for family, metrics in D_percent_metrics.items():
        for metric_name, da in metrics.items():
            series = weighted_tile_mean(da, domain_mask).load()
            for t, value in zip(pd.to_datetime(series.time.values), series.values):
                D_pct_rows.append({
                    "time": t,
                    "domain": domain_name,
                    "family": family,
                    "metric": metric_name,
                    "value": float(value) if np.isfinite(value) else np.nan,
                    "weighted": use_area_weights,
                    "formula": "100 * (DA - OL) / OL",
                })
analysisD_percent_timeseries = pd.DataFrame(D_pct_rows)
analysisD_percent_timeseries.to_csv(OUT_DIR / "analysisD_domain_monthly_timeseries_percent.csv", index=False)
display(analysisD_percent_timeseries.head(30))

plot_domain = "nh_seasonal_snow"
fig, axes = plt.subplots(3, 1, figsize=(13, 9.0), sharex=True)

# Top panel: original activity context.
activity_sub = analysisD_timeseries[(analysisD_timeseries["domain"] == plot_domain) & (analysisD_timeseries["family"] == "increment_activity")]
ax = axes[0]
for metric in ["snow_abs_netpack", "soil_water_abs_activity"]:
    sm = activity_sub[activity_sub["metric"] == metric]
    if not sm.empty:
        ax.plot(sm["time"], sm["value"], linewidth=1.2, label=metric_label(metric, include_units=True, context="monthly"))
ax2 = ax.twinx()
sm = activity_sub[activity_sub["metric"] == "RZMC_INC_RMS"]
if not sm.empty:
    ax2.plot(sm["time"], sm["value"], linewidth=1.1, linestyle="--", label=metric_label("RZMC_INC_RMS", include_units=True, context="monthly"), color="0.2")
ax.set_title("DA activity (native units; no OL baseline)")
ax.set_ylabel("Increment activity\n(kg m-2 month-1)")
ax2.set_ylabel("RZMC RMS\n(m3 m-3)")

# Middle/bottom panels: percent responses.
plot_sets_pct = [
    {
        "family": "hydrology_percent_response",
        "primary": ["pctTWLAND", "pctEVLAND", "pctTOTAL_RUNOFF"],
        "secondary": ["pctRZMC"],
        "title": "Hydrologic response (% change vs OL)",
        "ylabel": "Water response\n(% change vs OL)",
        "secondary_ylabel": "RZMC response\n(% change vs OL)",
    },
    {
        "family": "energy_percent_response",
        "primary": ["pctLHLAND", "pctSHLAND"],
        "secondary": ["pctEF"],
        "title": "Energy response (% change vs OL)",
        "ylabel": "Turbulent heat flux\n(% change vs OL)",
        "secondary_ylabel": "EF response\n(% change vs OL)",
    },
]
for ax, spec in zip(axes[1:], plot_sets_pct):
    sub = analysisD_percent_timeseries[(analysisD_percent_timeseries["domain"] == plot_domain) & (analysisD_percent_timeseries["family"] == spec["family"])]
    for metric in spec["primary"]:
        sm = sub[sub["metric"] == metric]
        if not sm.empty:
            ax.plot(sm["time"], sm["value"], linewidth=1.2, label=metric)
    ax2 = ax.twinx() if spec["secondary"] else None
    if ax2 is not None:
        for metric in spec["secondary"]:
            sm = sub[sub["metric"] == metric]
            if not sm.empty:
                ax2.plot(sm["time"], sm["value"], linewidth=1.1, linestyle="--", label=metric, color="0.2")
        ax2.axhline(0, color="0.45", linewidth=0.6, linestyle=":")
        ax2.set_ylabel(spec["secondary_ylabel"])
    ax.set_title(spec["title"])
    ax.set_ylabel(spec["ylabel"])
    handles, labels = ax.get_legend_handles_labels()
    if ax2 is not None:
        h2, l2 = ax2.get_legend_handles_labels()
        handles += h2
        labels += l2
    ax.legend(handles, labels, loc="upper left", fontsize=8, ncol=2)

for ax in axes:
    ax.axhline(0, color="0.2", linewidth=0.7)
    for period_key, period in PERIODS.items():
        ax.axvspan(pd.Timestamp(period["start"]), pd.Timestamp(period["end"]), color=period["color"], alpha=0.08)
    ax.grid(True, alpha=0.25)
axes[-1].set_xlabel("time")
fig.suptitle("Analysis D percent-change companion: monthly responses, NH seasonal-snow domain")
fig.tight_layout(rect=[0.035, 0, 0.965, 0.95])
fig.subplots_adjust(left=0.11, right=0.88, hspace=0.36, top=0.92)
savefig(fig, "analysisD_monthly_timeseries_nh_seasonal_snow_percent")
show_figure(fig)


## Analysis E: Water-Budget Sanity Check

This is a plausibility check, not formal closure. The residual below uses `P - E - runoff - WCHANGELAND`; interpretation depends on the exact `WCHANGELAND` sign convention.


### Figure Notes: Analysis E Water-Budget Sanity Check

The next cell creates `analysisE_water_budget_sanity_nh_seasonal_snow`. It is a monthly area-weighted domain-mean plausibility check for the NH seasonal-snow mask, not formal budget closure. Background shading marks observing-system periods.

The plotted response terms are `dE`, `dRUNOFF`, `dWCHANGELAND`, `dTWLAND`, and `dresidual_assuming_dP_zero`. `dE` is DA-OL evapotranspiration. `dRUNOFF` is DA-OL `RUNSURFLAND + BASEFLOWLAND`. `dWCHANGELAND` is DA-OL water-storage tendency. `dTWLAND` is DA-OL total water storage. Flux/tendency terms are monthly totals in `kg m-2 month-1` where applicable; `dTWLAND` is storage in `kg m-2`.

Precipitation is intentionally omitted because DA and OL are assumed conceptually to use the same precipitation forcing, so `dP = 0`. The residual is therefore `-(dE + dRUNOFF + dWCHANGELAND)`. Its sign and magnitude still depend on the exact `WCHANGELAND` convention and collection details. The remaining panels show increment context: `snow_net`, `snow_abs_netpack`, and `soil_water_abs_activity`, all in `kg m-2`.


In [ ]:
def monthly_water_budget_terms(experiment: str):
    p = monthly_model_var(experiment, "PRECTOTCORRLAND")
    e = monthly_model_var(experiment, "EVLAND")
    runoff = monthly_model_var(experiment, "RUNSURFLAND") + monthly_model_var(experiment, "BASEFLOWLAND")
    storage_tendency = monthly_model_var(experiment, "WCHANGELAND")
    twland = monthly_model_var(experiment, "TWLAND")
    return {
        "P": p,
        "E": e,
        "runoff": runoff,
        "WCHANGELAND": storage_tendency,
        "TWLAND": twland,
    }

E_rows = []
for domain_name, domain_mask in {"global_land": valid_land_mask, "nh_seasonal_snow": seasonal_snow_mask}.items():
    terms_ol = monthly_water_budget_terms("ol")
    terms_da = monthly_water_budget_terms("da")
    delta_terms = {
        "dE": terms_da["E"] - terms_ol["E"],
        "drunoff": terms_da["runoff"] - terms_ol["runoff"],
        "dWCHANGELAND": terms_da["WCHANGELAND"] - terms_ol["WCHANGELAND"],
        "dTWLAND": terms_da["TWLAND"] - terms_ol["TWLAND"],
    }
    # Precipitation should be common forcing for DA and OL. We therefore omit dP
    # from the plotted/check residual and treat it as conceptually zero here.
    delta_terms["dresidual_assuming_dP_zero"] = -(delta_terms["dE"] + delta_terms["drunoff"] + delta_terms["dWCHANGELAND"])
    for term, da in delta_terms.items():
        delta_series = weighted_tile_mean(da, domain_mask).load()
        for t, value in zip(pd.to_datetime(delta_series.time.values), delta_series.values):
            E_rows.append({
                "time": t,
                "domain": domain_name,
                "term": term,
                "value": float(value) if np.isfinite(value) else np.nan,
                "weighted": use_area_weights,
                "caveat": "Assumes DA and OL precipitation forcing are conceptually identical; dP intentionally omitted. Residual sign still depends on WCHANGELAND convention.",
            })
    for term_name, da in {"snow_net": CATCH["snow_net"], "snow_abs_netpack": CATCH["snow_abs_netpack"], "soil_water_abs_activity": CATCH["soil_water_abs_activity"]}.items():
        series = weighted_tile_mean(da, domain_mask).load()
        for t, value in zip(pd.to_datetime(series.time.values), series.values):
            E_rows.append({"time": t, "domain": domain_name, "term": term_name, "value": float(value) if np.isfinite(value) else np.nan, "weighted": use_area_weights, "caveat": "increment context"})
analysisE_monthly = pd.DataFrame(E_rows)
analysisE_monthly.to_csv(OUT_DIR / "analysisE_monthly_water_budget_sanity_timeseries.csv", index=False)
display(analysisE_monthly.head(30))

analysisE_period_summary = []
for period_key, period in PERIODS.items():
    sub = analysisE_monthly[(analysisE_monthly["time"] >= pd.Timestamp(period["start"])) & (analysisE_monthly["time"] <= pd.Timestamp(period["end"]))]
    summary = sub.groupby(["domain", "term"], dropna=False)["value"].agg(["mean", "median", "std", "count"]).reset_index()
    summary["period"] = period_key
    analysisE_period_summary.append(summary)
analysisE_period_summary = pd.concat(analysisE_period_summary, ignore_index=True)
analysisE_period_summary.to_csv(OUT_DIR / "analysisE_period_water_budget_sanity_summary.csv", index=False)
display(analysisE_period_summary.head(40))

plot_terms = ["dE", "drunoff", "dWCHANGELAND", "dTWLAND", "dresidual_assuming_dP_zero", "snow_net", "snow_abs_netpack", "soil_water_abs_activity"]
sub = analysisE_monthly[(analysisE_monthly["domain"] == "nh_seasonal_snow") & (analysisE_monthly["term"].isin(plot_terms))]
fig, axes = plt.subplots(4, 2, figsize=(13, 9.8), sharex=True)
for ax, term in zip(axes.ravel(), plot_terms):
    st = sub[sub["term"] == term]
    ax.plot(st["time"], st["value"], linewidth=1.1)
    ax.axhline(0, color="0.2", linewidth=0.7)
    for period_key, period in PERIODS.items():
        ax.axvspan(pd.Timestamp(period["start"]), pd.Timestamp(period["end"]), color=period["color"], alpha=0.08)
    ax.set_title(metric_label(term, include_units=True, context="monthly"), fontsize=9)
    ax.set_ylabel(metric_units(term, context="monthly") or "value")
    ax.grid(True, alpha=0.25)
fig.suptitle("Analysis E: water-budget sanity check, NH seasonal-snow domain (dP assumed zero)")
fig.tight_layout(rect=[0, 0, 1, 0.96])
savefig(fig, "analysisE_water_budget_sanity_nh_seasonal_snow")
show_figure(fig)


## Recommendation Scaffold

This final table ranks the strongest quick-look signals. It is meant to guide manuscript/supplement choices after inspecting the maps, binned summaries, anomaly controls, and water-budget sanity checks.


In [ ]:
recommendation_rows = []
recommendation_rows.append({
    "diagnostic": "Analysis A: snow carryover",
    "quick_signal": "Highest minus lowest MAM snow_abs_netpack bin for MJJ DA-OL RZMC magnitude",
    "value": high_low_signal(analysisA_binned_activity, "snow_abs_netpack_mam", "abs_drzmc_mjj"),
    "unit": "m3 m-3 absolute response",
    "candidate_level": "main or supplement if maps and bins are coherent",
    "interpretation_guardrail": "MODIS-only period supports snow-DA propagation, but not independent improvement.",
})
recommendation_rows.append({
    "diagnostic": "Analysis B: later SM-DA activity",
    "quick_signal": "SMAP-era highest minus lowest MAM snow_abs_netpack bin for MJJ RZMC_INC_RMS",
    "value": high_low_signal(analysisB_binned_activity, "snow_abs_netpack_mam", "rzmc_inc_rms_mjj", period="smap_era"),
    "unit": "m3 m-3 RMS activity",
    "candidate_level": "main or supplement only if anomaly control agrees",
    "interpretation_guardrail": "Higher activity may mean persistent snow-related hydrologic error, not necessarily worse DA.",
})
recommendation_rows.append({
    "diagnostic": "Analysis B anomaly control",
    "quick_signal": "SMAP-era within-tile anomaly: MAM snow_abs_netpack vs MJJ RZMC_INC_RMS",
    "value": high_low_signal(analysisB_binned_anomaly, "snow_abs_netpack_mam_anom", "rzmc_inc_rms_mjj_anom", period="smap_era"),
    "unit": "m3 m-3 RMS activity anomaly",
    "candidate_level": "critical robustness check",
    "interpretation_guardrail": "If this is flat while raw bins are strong, geography likely dominates the raw relationship.",
})
recommendation_rows.append({
    "diagnostic": "Analysis C1: warm energy partitioning",
    "quick_signal": "Highest minus lowest JJA DA-OL RZMC bin for JJA DA-OL EF",
    "value": high_low_signal(analysisC1_binned, "drzmc_jja", "def_jja"),
    "unit": "EF fraction",
    "candidate_level": "discussion/supplement if physically coherent with LE/H maps",
    "interpretation_guardrail": "Model-internal land-atmosphere coupling signal, not validation.",
})
recommendation_rows.append({
    "diagnostic": "Analysis E: water-budget sanity",
    "quick_signal": "Median NH seasonal-snow residual DA-OL over full record",
    "value": float(analysisE_monthly[(analysisE_monthly["domain"] == "nh_seasonal_snow") & (analysisE_monthly["term"] == "dresidual_assuming_dP_zero")]["value"].median()),
    "unit": "kg m-2 month-1",
    "candidate_level": "sanity check only",
    "interpretation_guardrail": "Precipitation is treated as common forcing and omitted from the residual; residual still depends on WCHANGELAND sign convention.",
})
recommendation = pd.DataFrame(recommendation_rows)
recommendation.to_csv(OUT_DIR / "monthly_synthesis_recommendation_quicklook.csv", index=False)
display(recommendation)

print("Notebook outputs written under", OUT_DIR)
print("Figures written under", FIG_DIR)
